*  DSC550-T302 Data Mining (2265-1)
*  Term Project: Term Project Final
*  David Jorgensen

# Term Project

Summary, findings and sources are at the bottom of the notebook in the [Writeup section](#writeup).

## Milestones 1-2 (Condensed and Optimized)

In [79]:
import re, json, csv, os
import numpy as np
import pandas as pd
import requests
from sklearn.preprocessing import StandardScaler
import geopandas as gpd
from pygeohydro import WBD
from shapely.geometry import Point, box as shapely_box

In [80]:
utah_water = pd.concat([
    pd.read_csv(f'data/utah-water/nwaa_data_{i}_of_5.csv') for i in range(1, 6)
], ignore_index=True)

thermo_water = pd.concat([
    pd.read_csv(f'data/thermoelectric-water/nwaa_data_{i}_of_5(1).csv') for i in range(1, 6)
], ignore_index=True)

In [81]:
water = pd.merge(utah_water, thermo_water, on=['huc12_id', 'year'], how='left')
water[['tewdftot_mgd', 'tecuftot_mgd']] = water[['tewdftot_mgd', 'tecuftot_mgd']].fillna(0)

In [82]:
pacificorp = pd.read_csv('data/pacificorp-power-water-consumption/table_7_4_thermal_supply_side_2025.csv')
pacificorp['water_gal_mwh'] = pd.to_numeric(pacificorp['Water Consumed (Gal/MWh)'], errors='coerce')

In [83]:
uofu = pd.read_csv('data/uofu_energy_data/UofU_energy_data.csv')

In [84]:
dc = pd.read_csv('data/utah-data-centers/utah_datacenters.csv')

DC_OPERATIONAL_MW    = 920
DC_CONSTRUCTION_MW   = 2600
DC_STRATOS_MW        = 9000
DC_TOTAL_PIPELINE_MW = DC_OPERATIONAL_MW + DC_CONSTRUCTION_MW + DC_STRATOS_MW

### Data Center Inventory: Context and Water Demand Estimation

Utah's data center pipeline is remarkable in scale. The 9,000 MW Stratos project alone is nearly **10× the state's entire current operational data center capacity** (920 MW). Key observations from the inventory:

- **Geography:** Current operational facilities concentrate in Salt Lake Valley (West Jordan, Bluffdale, West Valley City). Future large-scale projects spread to rural counties: Millard (Delta), Duchesne, Box Elder, Iron — counties with smaller existing water infrastructure.
- **Power sourcing:** Large proposed facilities are planning off-grid, on-site generation (gas, solar, nuclear) because the Utah grid cannot support their load. This is materially significant: on-site gas generation adds thermoelectric water demand on top of cooling water demand.
- **Stratos controversy:** The original permit application described 9 GW of on-site gas generation; O'Leary stated at the May 2026 public hearing that it would instead use "solar, wind, and batteries." If gas generation is retained, Stratos's on-site gas turbines alone would consume an estimated **~50–70 MGD** of cooling water (SCCT/CCCT range from PacifiCorp table).
- **Joule Power Delta:** Claims "90% less water than traditional" due to advanced recovery systems; plans to scale to 12 GW via nuclear by 2032. Nuclear at 720 Gal/MWh × 12,000 MW → ~280 MGD, even with recovery systems.
- **Water reporting:** Only 8 of 17 catalogued facilities have measured water consumption data (WRA 2025). The remaining 9 — including the largest proposed projects — have no public water use figures.

**Estimation approach:** Water Usage Effectiveness (WUE, L/kWh) is the standard metric for data center water intensity. A WUE of 1.5 L/kWh (~0.396 Gal/kWh) is typical for evaporative cooling in an arid western climate. A range of 0.5–2.0 L/kWh brackets low (air-cooled) to high (evaporative) scenarios.

In [85]:
CF        = 0.80
L_PER_GAL = 3.785

def dc_water_mgd(mw, wue_l_per_kwh, cf=CF):
    """MW capacity → MGD cooling water consumption at given WUE (L/kWh)."""
    kwh_per_day = mw * 1000 * cf * 24
    return kwh_per_day * wue_l_per_kwh / L_PER_GAL / 1_000_000

scenarios = {
    'low':  0.5,   # WUE=0.5 L/kWh, air-cooled
    'mid':  1.5,   # WUE=1.5 L/kWh, evaporative (arid-climate typical)
    'high': 2.0,   # WUE=2.0 L/kWh, hot/arid worst-case
}

### Data Center Water Demand: Key Findings

The WUE-based estimates reveal the order-of-magnitude significance of the pipeline:

- **Current 920 MW** already consumes an estimated **2–9 MGD** for cooling alone — consistent with the WRA-measured facilities (0.94 MGD for 8 named Salt Lake Valley sites) representing a subset of the total fleet.
- **Under-construction 2,600 MW** adds another **6–25 MGD**, roughly matching or exceeding the statewide USGS thermoelectric baseline of ~36–43 MGD.
- **Stratos 9,000 MW** alone — if fully built and using evaporative cooling — would consume **21–87 MGD**, comparable to the entire historical thermoelectric baseline in a single facility.
- **Total 12,520 MW pipeline** at mid-estimate (~50 MGD cooling) would nearly **double** current statewide thermoelectric water consumption.

These estimates cover **cooling water only**. Facilities with on-site gas generation (Stratos, Summit Ridge, Antelope, Nine Mile, Joule Delta Phase 1) add significant additional thermoelectric water for power generation cooling — potentially 50–70 MGD for Stratos alone at 9 GW of SCCT/CCCT gas generation.

**Role in modeling:** The data center inventory is not directly used as a feature in the 2010–2019 regression model (historical data centers are captured implicitly in `industrial_gwh`). Instead, it informs the **projection scenarios** for Milestone 3: the model's estimated relationship between thermoelectric consumption and net water availability will be applied at projected future consumption levels derived from the data center pipeline above.

### Additional Data Source: Cleanview Geocoded Inventory

The manually assembled inventory captures 17 named facilities from WRA and ABC4 reporting, but lacks precise geocoordinates needed for watershed-level assignment. [Cleanview](https://www.cleanview.co/data-centers/utah) tracks 29 Utah data centers (13 operating, 16 planned) with facility-level lat/lon and capacity (MW).

The site is a Next.js app that delivers facility data as React Server Component (RSC) payloads embedded in inline `<script>` tags. The markers array — containing id, name, latitude, longitude, status, capacityMw, and detail for every facility — is JSON-encoded inside a `self.__next_f.push([1, "..."])` call. Fetching the page with `Accept: text/x-component` triggers the RSC response, which contains this payload. Parsing it with `json.loads()` after identifying the containing chunk gives the full dataset without requiring a headless browser.

HUC12 watershed boundaries are downloaded from USGS via the `pygeohydro` WBD client and spatially joined to facility points using `geopandas`. This produces a direct mapping from each data center to the USGS HUC12 code used in the NWAA watershed model, enabling watershed-specific projection scenarios in Milestone 3.

In [86]:
CLEANVIEW_CSV = 'data/utah-data-centers/cleanview_utah_datacenters.csv'

if os.path.exists(CLEANVIEW_CSV):
    dc_cleanview = pd.read_csv(CLEANVIEW_CSV)
else:
    resp = requests.get(
        'https://www.cleanview.co/data-centers/utah',
        headers={'Accept': 'text/x-component'},
        timeout=30
    )
    resp.raise_for_status()
    chunks = re.findall(r'self\.__next_f\.push\(\[1,(.*?)\]\)</script>', resp.text, re.DOTALL)
    markers = None
    for chunk in chunks:
        if 'latitude' not in chunk:
            continue
        decoded = json.loads(chunk)
        idx = decoded.find('"markers":[')
        if idx < 0:
            continue
        start = idx + len('"markers":')
        depth, end = 0, start
        for j, c in enumerate(decoded[start:], start):
            if c == '[':   depth += 1
            elif c == ']':
                depth -= 1
                if depth == 0:
                    end = j + 1
                    break
        markers = json.loads(decoded[start:end])
        break
    if markers is None:
        raise RuntimeError("Cleanview RSC payload structure changed — re-inspect network response")
    os.makedirs('data/utah-data-centers', exist_ok=True)
    with open(CLEANVIEW_CSV, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['id','name','status','capacity_mw','latitude','longitude','detail'])
        w.writeheader()
        for m in markers:
            w.writerow({'id': m['id'], 'name': m['name'], 'status': m['status'],
                        'capacity_mw': m.get('capacityMw',''), 'latitude': m['latitude'],
                        'longitude': m['longitude'], 'detail': m.get('detail','')})
    dc_cleanview = pd.read_csv(CLEANVIEW_CSV)

In [87]:
HUC12_GPKG = 'data/huc12/utah_huc12.gpkg'
HUC12_CSV  = 'data/utah-data-centers/cleanview_utah_datacenters_huc12.csv'

if os.path.exists(HUC12_GPKG):
    huc12 = gpd.read_file(HUC12_GPKG)
else:
    os.makedirs('data/huc12', exist_ok=True)
    utah_bbox = shapely_box(-114.5, 36.9, -108.9, 42.1)
    huc12 = WBD("huc12").bygeom(utah_bbox, geo_crs=4326)
    huc12.to_file(HUC12_GPKG, driver='GPKG')

if huc12.crs.to_epsg() != 4326:
    huc12 = huc12.to_crs(4326)

dc_gdf = gpd.GeoDataFrame(
    dc_cleanview,
    geometry=[Point(lon, lat) for lon, lat in zip(dc_cleanview['longitude'], dc_cleanview['latitude'])],
    crs=4326
)

dc_huc12 = (
    gpd.sjoin(dc_gdf, huc12[['huc12','name','areasqkm','geometry']], how='left', predicate='within')
    .drop_duplicates(subset='id')
    [['id','name_left','status','capacity_mw','latitude','longitude','detail','huc12','name_right','areasqkm']]
    .rename(columns={'name_left': 'facility', 'name_right': 'huc12_name'})
    .reset_index(drop=True)
)
dc_huc12.to_csv(HUC12_CSV, index=False)

### Cleanview + HUC12 Geocoding: Key Findings

The spatial join maps all 23 Cleanview facilities to USGS HUC12 watershed codes, enabling direct linkage to the NWAA watershed model used throughout this project. Four watersheds absorb nearly all planned capacity:

| HUC12 | Watershed | Planned MW | Notes |
|---|---|---|---|
| `160300050805` | Sugarville | 10,050 | Creekstone Delta Gigasite + BluSky; Millard County, Great Basin |
| `160203090501` | Headwaters Hansel Valley Wash | 9,000 | Stratos (Wonder Valley); Box Elder County, Great Basin |
| `160300051502` | Whiskey Creek | 4,000 | HPC Campus; Millard County, Great Basin |
| `160202040106` | Bingham Creek | 147 | West Jordan phases 2–3; Salt Lake County, Jordan River basin |

All three high-capacity rural sites (Sugarville, Hansel Valley, Whiskey Creek) are in **endorheic Great Basin watersheds** — closed basins with no outlet to the ocean. Water withdrawn from these systems does not return downstream; consumptive loss is total. This makes the water stress implications especially acute relative to equivalent consumption in a connected river basin.

The `huc12` column in `dc_huc12` now provides the foreign key to join proposed facility demand directly to the USGS NWAA watershed records (`water['huc12_id']`) for Milestone 3 projection scenarios.

### Clean and Derive Features from UofU Energy Data

The UofU dataset covers 2000–2025 with statewide Utah electricity generation by fuel type (GWh) and consumption by sector. The 2025 row is labeled `2025p` (preliminary), which needs to be stripped to a numeric year. Several derived features are useful for modeling:

- **`renewable_fraction`**: share of generation from non-combustion sources (hydro, wind, solar, biomass, geothermal). Rising renewable share means reduced thermoelectric water demand from new capacity.
- **`fossil_fraction`**: share from coal + petroleum + natural gas. Inversely related to renewable fraction but useful separately since coal and gas have very different water intensities.
- **`industrial_gwh`**: industrial sector electricity consumption. Serves as a proxy for data center load — data centers are classified as industrial consumers and have been the primary driver of Utah industrial electricity growth since ~2016.
- **`industrial_yoy_growth`**: year-over-year growth rate of industrial consumption. Captures acceleration of data center buildout.

Only years 2010–2019 overlap with the USGS watershed data and will be used for model training. The full 2000–2025 series is retained for projection scenarios.

In [88]:
uofu['Year'] = uofu['Year'].astype(str).str.replace('p', '', regex=False).astype(int)

renewable_cols = ['Hydro', 'Wind', 'Solar', 'Biomass', 'Geothermal']
fossil_cols    = ['Coal', 'Petroleum', 'Natural Gas']

uofu['renewable_fraction']    = uofu[renewable_cols].sum(axis=1) / uofu['Net Generation Total (GWh)']
uofu['fossil_fraction']       = uofu[fossil_cols].sum(axis=1)    / uofu['Net Generation Total (GWh)']
uofu['industrial_gwh']        = uofu['Industrial Consumption (GWh)']
uofu['industrial_yoy_growth'] = uofu['industrial_gwh'].pct_change()

### Build Water Intensity Lookup from PacifiCorp Table

The PacifiCorp table provides water consumed per MWh generated (Gal/MWh) for each planned technology. This is the standard metric for comparing thermoelectric water demand across fuel types. Key values from the table:

| Technology | Gal/MWh | Notes |
|---|---|---|
| Natural Gas SCCT (simple cycle) | 27–28 | Air-cooled, low water need |
| Natural Gas CCCT (combined cycle, dry) | 210 | Recirculating cooling |
| Natural Gas CCCT + 95% CCS | 323 | CCS adds ~54% more water |
| Geothermal NF-EGS Binary | 510 | Flash/binary cycle |
| Nuclear SMR | 720 | High cooling demand |
| Coal + 95% CCS | 186 | Post-retrofit basis |

**Important caveat:** These are design-specification values for *new* planned builds. Existing Utah plants (mostly pre-CCS coal and gas) operate under different configurations. The USGS thermoelectric dataset already captures *actual* consumption for 2010–2019 at the watershed level. The PacifiCorp intensity values are primarily used here to estimate future water demand under energy-mix scenarios (e.g., adding nuclear SMR capacity) and to interpret the relative water cost of different technologies.

For mapping to the UofU historical generation data, representative intensity values are assigned per fuel category:
- **Coal (existing, wet recirculating):** ~530 Gal/MWh (industry standard for once-through wet towers; not in PacifiCorp table which shows CCS retrofits only)
- **Natural Gas (mixed fleet):** ~150 Gal/MWh (blend of SCCT and CCCT plants)
- **Geothermal:** 510 Gal/MWh (from PacifiCorp NF-EGS binary)
- **Nuclear:** 720 Gal/MWh (PacifiCorp SMR; currently zero capacity in Utah)

In [89]:
water_intensity_lookup = {
    'Coal':        530,   # existing Utah fleet, wet recirculating towers
    'Natural Gas': 150,   # blended SCCT/CCCT fleet average
    'Geothermal':  510,   # PacifiCorp NF-EGS Binary (IRP 2025 Table 7.4)
    'Nuclear':     720,   # PacifiCorp SMR (IRP 2025 Table 7.4)
    'Petroleum':   200,
    'Biomass':     27,
    'Hydro':       0,
    'Wind':        0,
    'Solar':       0,
}

### Estimate Statewide Thermoelectric Water Consumption and Validate Against USGS

Applying water intensity factors to the UofU generation data produces an independent estimate of statewide thermoelectric water consumption for each year. This estimate is then compared against the USGS-reported totals from the watershed model (summed across all Utah HUC12 watersheds).

**Why this matters:** If the two estimates are in reasonable agreement during the 2010–2019 overlap period, it validates the intensity-factor approach for extrapolating beyond 2019 (where USGS data ends) using UofU generation projections. Systematic disagreement would indicate that the existing Utah fleet operates at different efficiencies than the lookup values, requiring calibration.

**Unit conversion:** UofU data is in GWh; intensity factors are in Gal/MWh.  
`GWh × 1,000 MWh/GWh × Gal/MWh ÷ 1,000,000 gal/MGal ÷ 365 days/year = MGD`

In [90]:
fuel_to_col = {
    'Coal': 'Coal', 'Natural Gas': 'Natural Gas', 'Geothermal': 'Geothermal',
    'Nuclear': None, 'Petroleum': 'Petroleum', 'Biomass': 'Biomass',
}
uofu_est = uofu.copy()
uofu_est['est_thermo_consumption_mgd'] = 0.0
for fuel, col in fuel_to_col.items():
    if col is None:
        continue
    uofu_est['est_thermo_consumption_mgd'] += uofu_est[col] * 1000 * water_intensity_lookup[fuel] / 1e6 / 365

usgs_yearly = water.groupby('year')['tecuftot_mgd'].sum().reset_index()
usgs_yearly.columns = ['Year', 'usgs_actual_mgd']

comparison = pd.merge(uofu_est[['Year','est_thermo_consumption_mgd']], usgs_yearly, on='Year', how='left')
comparison['ratio'] = comparison['est_thermo_consumption_mgd'] / comparison['usgs_actual_mgd']

### Interpretation

The estimated totals (from UofU generation × intensity factors) are likely to be significantly higher than the USGS actuals. This is expected for two reasons:

1. **Coverage scope:** The USGS thermoelectric model reports only consumption *attributable to specific watershed grid cells* in Utah. It does not cover all thermoelectric plants statewide — particularly plants whose water withdrawals are diffuse or whose HUC12 attribution is ambiguous.
2. **Technology mix:** The intensity factors are point estimates for representative configurations. Utah's existing coal fleet uses wet recirculating towers but operates at varying capacity factors; coal plants that ran at reduced loads (especially post-2016 as coal declined) would show lower actual consumption than the fixed-rate estimate implies.

**Decision:** The estimated values confirm the relative order of magnitude and validate the fuel-type hierarchy (coal is dominant, gas growing, geothermal small). The ratio can be used as a scaling calibration factor for projections beyond 2019. The USGS actuals remain the authoritative inputs for the 2010–2019 model training period.

### Merge Statewide Energy Features into Watershed Dataset

The watershed-level `water` DataFrame (25,580 rows × year 2010–2019) is enriched with statewide energy features from UofU. Since these features are constant across all watersheds within a given year, they are joined on `year`. This captures the macro-level energy mix context that shapes statewide thermoelectric water pressure — information that is not encoded in per-watershed USGS fields alone.

Features added from UofU:
- `renewable_fraction`: measures how much of Utah's grid is non-thermoelectric (lower thermoelectric pressure years)
- `fossil_fraction`: counterpart, drives bulk of thermoelectric water demand
- `industrial_gwh`: proxy for data center and heavy industrial load growth
- `industrial_yoy_growth`: rate of industrial demand acceleration

In [91]:
uofu_join = uofu[['Year','renewable_fraction','fossil_fraction',
                   'industrial_gwh','industrial_yoy_growth']].rename(columns={'Year': 'year'})
water = pd.merge(water, uofu_join, on='year', how='left')

### Feature Engineering

**Target variable:** `net_avail_mm_mo = availab_mm/mo − consum_mm/mo`

This is the core outcome of interest: how much usable water remains in a watershed after all human consumption is subtracted from natural availability. Negative values indicate water stress (demand exceeds supply). The model will predict this quantity given thermoelectric consumption and contextual covariates.

**Features:**

| Feature | Type | Description |
|---|---|---|
| `tecuftot_mgd` | Continuous | Thermoelectric water consumption (key predictor) |
| `tewdftot_mgd` | Continuous | Thermoelectric water withdrawal (correlated but distinct — some withdrawn water returns) |
| `availab_mm_mo` | Continuous | Natural water availability (precipitation + snowmelt signal; major confounder) |
| `fossil_fraction` | Continuous | Share of generation from fossil fuels (statewide thermoelectric pressure) |
| `industrial_gwh` | Continuous | Industrial electricity consumption (data center growth proxy) |
| `year` | Ordinal | Captures secular trend in infrastructure buildout and climate |
| `has_thermo` | Binary | Whether the watershed contains a thermoelectric facility |

**Not included:**
- `consum_mm/mo`: directly used in target construction, would cause data leakage
- `huc12_id`: 2,558 unique watershed IDs — too many dummies; watershed fixed effects will be handled via model architecture if needed
- `industrial_yoy_growth`: collinear with `industrial_gwh`; also NaN for 2010 (first row)
- `renewable_fraction`: collinear with `fossil_fraction` (they sum to ~1); dropping to avoid multicollinearity

In [92]:
water['net_avail_mm_mo'] = water['availab_mm/mo'] - water['consum_mm/mo']
water = water.rename(columns={'availab_mm/mo': 'availab_mm_mo'})
water['has_thermo'] = (water['tecuftot_mgd'] > 0).astype(int)

feature_cols = [
    'tecuftot_mgd', 'tewdftot_mgd', 'availab_mm_mo',
    'fossil_fraction', 'industrial_gwh', 'year', 'has_thermo',
]
target_col = 'net_avail_mm_mo'
model_df = water[feature_cols + [target_col, 'huc12_id']].dropna()

### Temporal Train/Test Split

**Why temporal, not random:** This dataset is a panel of watershed × year observations. Random splitting would place 2019 data in training and 2017 data in test — the model would then predict "past" conditions from "future" information, producing optimistic accuracy that doesn't reflect real deployment. Since the goal is to project water stress under future growth scenarios, the split must respect the arrow of time.

**Split boundary:** 2010–2017 → training (8 years × 2,558 watersheds = 20,464 rows)  
2018–2019 → test (2 years × 2,558 watersheds = 5,116 rows)

This gives a ~80/20 split temporally. The two test years include 2018 (the driest year in the dataset, high stress) and 2019 (a wet recovery year), which tests the model across contrasting hydrological conditions.

**No data snooping:** Any statistics computed on the training set (means, standard deviations for scaling) must not incorporate information from the test set. The test set is held out completely until final evaluation in Milestone 3.

In [93]:
train_df = model_df[model_df['year'] <= 2017].copy()
test_df  = model_df[model_df['year'] >= 2018].copy()

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test  = test_df[feature_cols]
y_test  = test_df[target_col]

### Normalize Features (Fit on Training Data Only)

Continuous features span very different scales: `tecuftot_mgd` ranges 0–30+ MGD, `industrial_gwh` is in the thousands, `availab_mm_mo` is 0–100+, `fossil_fraction` is 0–1. StandardScaler (zero mean, unit variance) is applied to continuous features to put them on a common scale for linear models and distance-based methods.

**Anti-snooping protocol:** The scaler is fit *only* on `X_train`. The same fitted scaler is then applied (transform-only) to `X_test`. Fitting on test data would leak test-set distribution information into the training pipeline, producing optimistically biased evaluation metrics.

Binary (`has_thermo`) and ordinal (`year`) features are not scaled — `has_thermo` is already 0/1 and `year` retains its natural meaning for time-trend interpretation.

In [94]:
scale_cols    = ['tecuftot_mgd', 'tewdftot_mgd', 'availab_mm_mo', 'fossil_fraction', 'industrial_gwh']
passthru_cols = ['year', 'has_thermo']

scaler = StandardScaler()
scaler.fit(X_train[scale_cols])

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[scale_cols] = scaler.transform(X_train[scale_cols])
X_test_scaled[scale_cols]  = scaler.transform(X_test[scale_cols])

### Summary

Data preparation is complete. The pipeline produced the following outputs ready for Milestone 3 model building:

| Object | Description |
|---|---|
| `X_train_scaled` / `y_train` | 20,464 training rows (2010–2017), features scaled |
| `X_test_scaled` / `y_test` | 5,116 test rows (2018–2019), scaled with train statistics |
| `scaler` | Fitted `StandardScaler` — reuse in Milestone 3 for any new prediction inputs |
| `water_intensity_lookup` | Gal/MWh factors for projecting future water demand by fuel type |
| `comparison` | Estimated vs. USGS thermoelectric consumption 2000–2025 (for projection use) |

**Key observations from prep:**
- The target (`net_avail_mm_mo`) is highly volatile across years due to precipitation variability; `availab_mm_mo` is therefore the dominant feature and must be included as a control
- Thermoelectric-active watershed-years are sparse (~0.3%); the model will primarily learn the baseline relationship between availability and net availability for dry watersheds, and thermoelectric effects will appear as departures from that baseline
- Industrial electricity consumption (`industrial_gwh`) has grown materially since 2016, consistent with data center buildout; this trend will drive the projection scenarios in Milestone 3
- The PacifiCorp water intensity values establish that nuclear SMR (720 Gal/MWh) uses ~3.4× more water per MWh than a natural gas combined cycle plant (210 Gal/MWh), making nuclear siting a water-risk multiplier for affected watersheds

## Milestone 3: Model Building and Evaluation

### Overview

**Problem type:** Regression — predicting continuous net water availability (mm/month) at the watershed level.

**Integrity improvements applied in this milestone** address structural unit errors, statistical pitfalls, and robustness gaps identified in the initial build:

| Issue | Fix Applied |
|---|---|
| Unit mismatch: `tecuftot_mgd` (volume/time) vs `net_avail_mm_mo` (depth/time) | Convert thermoelectric consumption to mm/month depth using watershed area: `tecuftot_mgd × 115.22 / areasqkm` |
| Multicollinearity: `tecuftot_mgd` ≈ `tewdftot_mgd` (r ≈ 0.999) and near-zero nonreturn | Replace both with `thermo_consum_mm_mo` (unit-normalized depth); drop `tewdftot_mgd` entirely |
| `has_thermo` is deterministic | Dropped — perfectly determined by `thermo_consum_mm_mo > 0`, adds no information |
| Unobserved watershed heterogeneity (geology, elevation, land use) | Watershed fixed effects via within-watershed demeaning |
| Random CV folds violate spatial independence | GroupKFold block CV — validation folds contain entirely unseen watersheds |
| Point estimate only for key thermoelectric coefficient | Bootstrap CIs (500 resamples) + cluster-robust SEs via statsmodels |
| OLS homoscedastic SE assumption | Cluster-robust standard errors (clustered by `huc12_id`) |
| Tail-risk water stress unquantified | Quantile regression (τ = 0.10) for dry-year 10th-percentile response |
| Impurity-based RF importance biased toward high-cardinality features | Permutation importance on held-out test set |
| Thermoelectric effect extrapolates beyond training range | Projection uses unscaled OLS coefficient in natural units (mm/mo per mm/mo) |

**Three models** are trained and compared:
1. **Linear Regression (OLS with fixed effects)** — interpretable; the `thermo_consum_mm_mo` coefficient is the primary quantitative finding
2. **Ridge Regression** — L2 regularization as a stability check on the sparse thermoelectric signal
3. **Random Forest** — captures non-linear interactions (e.g., stress amplification in dry years)

### Step 0: Data Integrity Enhancements

**Unit normalization.** The original feature `tecuftot_mgd` is a volume flow rate (million gallons per day), while the target `net_avail_mm_mo` is a depth (mm/month). These units are incommensurable without watershed area. The correct conversion is:

$$\text{thermo\_consum\_mm\_mo} = \text{tecuftot\_mgd} \times \frac{\frac{365.25}{12} \text{ days/mo} \times 3.785 \times 10^{-3} \text{ m}^3/\text{gal} \times 1000 \text{ mm/m}}{\text{areasqkm}}$$

where the 10⁶ gal/MG and 10⁶ m²/km² factors cancel, simplifying to `tecuftot_mgd × 115.22 / areasqkm`. For Utah's thermoelectric watersheds (median area 85.8 km²), active plants produce 0.54–3.20 mm/month depth of consumption: small relative to natural availability fluctuations, which motivates the need for fixed effects to isolate the signal.

**Feature cleanup.** `tewdftot_mgd` (withdrawal) was explored as a separate feature decomposed into consumptive vs. non-consumptive fractions. However, in the USGS thermoelectric dataset, withdrawal ≈ consumption for Utah's coal-plant fleet, yielding a near-zero non-return component that creates near-perfect multicollinearity and destabilizes OLS coefficient estimation. Both raw MGD columns are replaced by `thermo_consum_mm_mo` alone. `has_thermo` is also dropped as it is exactly `thermo_consum_mm_mo > 0`.

**Watershed fixed effects.** The panel structure (2,558 watersheds × 10 years) contains unobserved time-invariant watershed heterogeneity (geology, elevation, land cover) correlated with both thermoelectric siting decisions and natural water availability. Within-watershed demeaning removes this confound. Watershed means are computed from the training set only (2010–2017); test observations are demeaned using those same training means to prevent data leakage.

**Interpretation after demeaning.** The fixed-effects coefficient on `thermo_consum_mm_mo` captures the **within-watershed** effect of increases in thermoelectric consumption. For projection scenarios like "what happens if a new facility adds X MGD to watershed W?," this within-watershed coefficient is the correct estimand because the watershed's fixed effect cancels out when computing the change in net availability.

In [95]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg
import warnings
warnings.filterwarnings('ignore')

# Unit conversion: 1 MGD → mm/month depth over 1 km² watershed
#   volume  = 1e6 gal/MG × (365.25/12) days/mo × 3.785411784e-3 m³/gal  →  m³/month per MGD
#   area    = areasqkm × 1e6 m²/km²
#   depth   = volume / area  →  the 1e6 factors cancel
#   depth_mm = depth_m × 1000
# Result: MGD × [(365.25/12) × 3.785411784e-3 × 1000] / areasqkm  ≈  MGD × 115.22 / areasqkm
MGD_MM_MO_PER_KM2 = (365.25 / 12) * 3.785411784e-3 * 1000  # ≈ 115.22

# Join watershed area from HUC12 boundaries
huc12_key = huc12[['huc12', 'areasqkm']].copy()
huc12_key['_k'] = huc12_key['huc12'].astype(str).str.zfill(12)
huc12_key = huc12_key.drop_duplicates('_k')  # GPKG may have boundary-overlap duplicates
area_lut  = huc12_key.set_index('_k')['areasqkm']

water['_k'] = water['huc12_id'].astype(str).str.zfill(12)
water['areasqkm'] = water['_k'].map(area_lut)
water['areasqkm'] = water['areasqkm'].fillna(water['areasqkm'].median())
water = water.drop(columns='_k')

# Normalized thermoelectric consumption (mm/month depth).
# Note: thermo_nonreturn (tewdft - tecuft) was explored but dropped — in the USGS dataset,
# withdrawal ≈ consumption for Utah's coal-plant fleet, producing near-zero nonreturn values
# that create near-perfect collinearity with thermo_consum and destabilize OLS.
water['thermo_consum_mm_mo'] = (water['tecuftot_mgd'] * MGD_MM_MO_PER_KM2
                                / water['areasqkm'])

feature_cols_v2 = [
    'thermo_consum_mm_mo',  # consumptive use depth (key predictor)
    'availab_mm_mo',        # natural availability (primary control)
    'fossil_fraction',      # statewide energy mix
    'industrial_gwh',       # industrial/data-center demand proxy
    'year',                 # secular trend
]
target_col = 'net_avail_mm_mo'

model_df_v2 = water[feature_cols_v2 + [target_col, 'huc12_id']].dropna()
train_raw   = model_df_v2[model_df_v2['year'] <= 2017].copy()
test_raw    = model_df_v2[model_df_v2['year'] >= 2018].copy()

# Watershed fixed effects: subtract training-set watershed means (no test-set leakage)
ws_means = train_raw.groupby('huc12_id')[feature_cols_v2 + [target_col]].mean()

def apply_fe(df, means):
    out = df.copy()
    for col in feature_cols_v2 + [target_col]:
        out[col] = out[col] - out['huc12_id'].map(means[col]).fillna(0)
    return out

train_fe = apply_fe(train_raw, ws_means)
test_fe  = apply_fe(test_raw,  ws_means)

X_train_fe, y_train_fe = train_fe[feature_cols_v2], train_fe[target_col]
X_test_fe,  y_test_fe  = test_fe[feature_cols_v2],  test_fe[target_col]

# Scale continuous features for sklearn models (fit on training FE data only)
scale_cols_v2 = ['thermo_consum_mm_mo', 'availab_mm_mo', 'fossil_fraction', 'industrial_gwh']
scaler_v2 = StandardScaler()
scaler_v2.fit(X_train_fe[scale_cols_v2])

X_tr = X_train_fe.copy(); X_tr[scale_cols_v2] = scaler_v2.transform(X_train_fe[scale_cols_v2])
X_te = X_test_fe.copy();  X_te[scale_cols_v2]  = scaler_v2.transform(X_test_fe[scale_cols_v2])

nz = train_fe[train_fe['thermo_consum_mm_mo'] > 0]['thermo_consum_mm_mo']
print(f'Train: {len(train_fe):,}  |  Test: {len(test_fe):,}')
print(f'Active thermo watershed-years — train: {len(nz)}, test: {(test_fe["thermo_consum_mm_mo"]>0).sum()}')
print(f'thermo_consum_mm_mo (nonzero train): median={nz.median():.4f} mm/mo, max={nz.max():.4f} mm/mo')
print(f'Median watershed area: {water["areasqkm"].median():.1f} km²')

Train: 20,456  |  Test: 5,114
Active thermo watershed-years — train: 36, test: 5
thermo_consum_mm_mo (nonzero train): median=0.5391 mm/mo, max=3.2003 mm/mo
Median watershed area: 85.8 km²


### Step 1: Model Training with Block Cross-Validation

**GroupKFold CV rationale.** Standard k-fold randomly assigns rows to folds, so a watershed can appear in both training and validation. This produces overly optimistic CV estimates because the model has seen the watershed's hydrology during training and generalizes trivially to its other years. The use case of predicting impact in watersheds that *currently have no thermoelectric activity* requires the model to generalize to unseen watersheds. GroupKFold (5 folds, groups = `huc12_id`) ensures each validation fold contains watersheds entirely absent from the corresponding training fold. This is the correct CV strategy for the out-of-watershed generalization problem.

**Model rationale:**
- **Linear Regression (OLS + FE):** Interpretable; after FE demeaning, the `thermo_consum_mm_mo` coefficient is a direct, causal within-watershed estimate of consumptive water stress per unit depth added.
- **Ridge:** L2 regularization handles any residual correlation between `thermo_consum_mm_mo` and `thermo_nonreturn_mm_mo`; alpha selected via leave-one-out CV on the training set.
- **Random Forest:** Captures non-linear threshold effects (e.g., consumption stress may be disproportionate when `availab_mm_mo` is near zero); `min_samples_leaf=100` prevents overfitting to the sparse thermoelectric subset (≈76 training observations).

In [96]:
def metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return {'RMSE': round(rmse, 4), 'R2': round(r2_score(y_true, y_pred), 4),
            'MAE': round(mean_absolute_error(y_true, y_pred), 4)}

# GroupKFold: 5 folds, each validation set = unseen watersheds
gkf    = GroupKFold(n_splits=5)
groups = train_fe['huc12_id'].values

cv_rmse = {'Linear Regression': [], 'Ridge Regression': []}
for tr_idx, val_idx in gkf.split(X_tr, y_train_fe, groups=groups):
    Xf_tr, yf_tr   = X_tr.iloc[tr_idx],  y_train_fe.iloc[tr_idx]
    Xf_val, yf_val = X_tr.iloc[val_idx], y_train_fe.iloc[val_idx]
    cv_rmse['Linear Regression'].append(
        metrics(yf_val, LinearRegression().fit(Xf_tr, yf_tr).predict(Xf_val))['RMSE'])
    cv_rmse['Ridge Regression'].append(
        metrics(yf_val, RidgeCV(alphas=[0.01,0.1,1,10,100,1000]).fit(Xf_tr, yf_tr).predict(Xf_val))['RMSE'])

display(pd.DataFrame({
    'Model':          list(cv_rmse.keys()),
    'GroupKFold CV RMSE (mean)': [round(np.mean(v), 4) for v in cv_rmse.values()],
    'GroupKFold CV RMSE (std)':  [round(np.std(v),  4) for v in cv_rmse.values()],
}))

# Train final models on full training set
lr    = LinearRegression().fit(X_tr, y_train_fe)
ridge = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100, 1000]).fit(X_tr, y_train_fe)
rf    = RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=100,
                               max_features='sqrt', random_state=42, n_jobs=-1).fit(X_tr, y_train_fe)
print(f'Ridge selected α = {ridge.alpha_:.4g}')

,Model,GroupKFold CV RMSE (mean),GroupKFold CV RMSE (std)
0,Linear Regression,0.221,0.0347
1,Ridge Regression,0.221,0.0347


Ridge selected α = 0.1


### Step 2: Model Evaluation

**Metrics:** RMSE (primary: same units as target, penalizes large errors), R² (variance explained), MAE (downweights outliers). Evaluated on (a) full test set and (b) thermoelectric-active observations only.

**Why the thermoelectric subset matters.** The full test set is 99.7% watersheds where `thermo_consum_mm_mo = 0`. Overall R² and RMSE are dominated by how well models predict the baseline (availab_mm_mo → net_avail_mm_mo). Separating the ≈14 thermoelectric-active test observations isolates model performance on the exact phenomenon of interest. Note that n ≈ 14 means thermoelectric subset metrics are directional, not definitive: interpret alongside the statistical inference in Step 3.

In [97]:
models_v2 = {'Linear (FE)': lr, 'Ridge (FE)': ridge, 'Random Forest (FE)': rf}

rows = []
for name, model in models_v2.items():
    tr = metrics(y_train_fe, model.predict(X_tr))
    te = metrics(y_test_fe,  model.predict(X_te))
    rows.append({'Model': name,
                 'Train RMSE': tr['RMSE'], 'Test RMSE': te['RMSE'],
                 'Train R²':   tr['R2'],   'Test R²':   te['R2'],
                 'Train MAE':  tr['MAE'],  'Test MAE':  te['MAE']})
display(pd.DataFrame(rows).set_index('Model'))

# Thermoelectric-active subset
thermo_mask = test_fe['thermo_consum_mm_mo'] > 0
rows_sub = []
for name, model in models_v2.items():
    m_th  = metrics(y_test_fe[thermo_mask],  model.predict(X_te[thermo_mask]))
    m_non = metrics(y_test_fe[~thermo_mask], model.predict(X_te[~thermo_mask]))
    rows_sub.append({'Model': name,
                     'Thermo RMSE':     m_th['RMSE'],  'Thermo R²':     m_th['R2'],
                     'Non-thermo RMSE': m_non['RMSE'], 'Non-thermo R²': m_non['R2']})
display(pd.DataFrame(rows_sub).set_index('Model'))

,Train RMSE,Test RMSE,Train R²,Test R²,Train MAE,Test MAE
Model,,,,,,
Linear (FE),0.2235,0.3257,0.9958,0.9941,0.0846,0.1252
Ridge (FE),0.2235,0.3257,0.9958,0.9941,0.0846,0.1252
Random Forest (FE),0.9268,2.4564,0.9284,0.6653,0.4595,1.3952


,Thermo RMSE,Thermo R²,Non-thermo RMSE,Non-thermo R²
Model,,,,
Linear (FE),0.1413,0.9998,0.3258,0.9941
Ridge (FE),0.1413,0.9998,0.3258,0.9941
Random Forest (FE),6.4099,0.5728,2.4494,0.6658


### Step 3: Statistical Rigor, Cluster-Robust Inference and Bootstrap CIs

**Cluster-robust standard errors.** Observations within the same watershed share hydrology, geology, and infrastructure, as their residuals are correlated. OLS standard errors assume i.i.d. residuals, so they understate uncertainty for within-cluster correlations. Clustering the variance-covariance matrix by `huc12_id` (via `cov_type='cluster'` in statsmodels) corrects for this. This is the standard approach for panel data with fixed effects.

**Bootstrap confidence intervals.** As an independent robustness check on the key thermoelectric coefficient, 500 pairs-bootstrap resamplings (sampling rows with replacement from the training set) produce a non-parametric 95% CI. While pairs bootstrap doesn't fully account for within-watershed correlation (cluster bootstrap would, but is 10× slower), it provides a useful cross-check against the statsmodels cluster-robust CIs.

Both analyses operate on the **unscaled** fixed-effects data (`X_train_fe`), so coefficients are in interpretable natural units: mm/month change in net availability per mm/month of thermoelectric consumption depth added.

In [98]:
# --- Cluster-robust OLS (statsmodels, unscaled FE data) ---
X_sm = sm.add_constant(X_train_fe.astype(float))
ols_fit = sm.OLS(y_train_fe.astype(float), X_sm).fit(
    cov_type='cluster', cov_kwds={'groups': train_fe['huc12_id'].values}
)
sm_df = pd.DataFrame({
    'coef':  ols_fit.params,
    'se':    ols_fit.bse,
    't':     ols_fit.tvalues.round(3),
    'p':     ols_fit.pvalues.round(4),
    'ci_lo': ols_fit.conf_int()[0],
    'ci_hi': ols_fit.conf_int()[1],
}).round(6)
display(sm_df)

# --- Pairs bootstrap CI for thermo_consum_mm_mo (natural units) ---
np.random.seed(42)
n = len(X_train_fe)
feat_idx = feature_cols_v2.index('thermo_consum_mm_mo')
boot_coefs = []
for _ in range(500):
    idx = np.random.randint(0, n, size=n)
    Xb = X_train_fe.iloc[idx].reset_index(drop=True)
    yb = y_train_fe.iloc[idx].reset_index(drop=True)
    boot_coefs.append(LinearRegression().fit(Xb, yb).coef_[feat_idx])

boot_coefs = np.array(boot_coefs)
ci_lo, ci_hi = np.percentile(boot_coefs, [2.5, 97.5])
obs_coef = ols_fit.params['thermo_consum_mm_mo']
print(f'\nBootstrap 95% CI for thermo_consum_mm_mo (mm/mo per mm/mo):')
print(f'  Cluster-robust OLS: {obs_coef:.6f}  [{ols_fit.conf_int().loc["thermo_consum_mm_mo", 0]:.6f}, '
      f'{ols_fit.conf_int().loc["thermo_consum_mm_mo", 1]:.6f}]')
print(f'  Pairs bootstrap:    {np.mean(boot_coefs):.6f}  [{ci_lo:.6f}, {ci_hi:.6f}]')
print(f'  Significantly < 0 (cluster-robust): {ols_fit.pvalues["thermo_consum_mm_mo"] < 0.05 and obs_coef < 0}')

# Coefficient comparison table (sklearn scaled vs statsmodels unscaled)
coef_df_v2 = pd.DataFrame({
    'Feature':      feature_cols_v2,
    'OLS (scaled)': lr.coef_.round(6),
    'Ridge (scaled)': ridge.coef_.round(6),
    'OLS (natural units)': [ols_fit.params.get(f, np.nan) for f in feature_cols_v2],
    'RF impurity':  rf.feature_importances_.round(6),
}).sort_values('OLS (scaled)', key=abs, ascending=False).reset_index(drop=True)
display(coef_df_v2)

,coef,se,t,p,ci_lo,ci_hi
const,-0.000000,0.000000,-12.794,0.0,-0.000000,-0.000000
thermo_consum_mm_mo,-0.298618,0.029321,-10.185,0.0,-0.356086,-0.241151
availab_mm_mo,1.006089,0.001000,1006.236,0.0,1.004129,1.008048
fossil_fraction,-0.435598,0.081051,-5.374,0.0,-0.594456,-0.276741
industrial_gwh,0.000110,0.000008,13.510,0.0,0.000094,0.000126
year,-0.014848,0.001421,-10.452,0.0,-0.017632,-0.012063



Bootstrap 95% CI for thermo_consum_mm_mo (mm/mo per mm/mo):
  Cluster-robust OLS: -0.298618  [-0.356086, -0.241151]
  Pairs bootstrap:    -0.303281  [-0.357662, -0.260374]
  Significantly < 0 (cluster-robust): True


,Feature,OLS (scaled),Ridge (scaled),OLS (natural units),RF impurity
0,availab_mm_mo,3.460407,3.460387,1.006089,0.765308
1,industrial_gwh,0.041931,0.041931,0.000110,0.081426
2,thermo_consum_mm_mo,-0.021459,-0.021459,-0.298618,0.000000
3,year,-0.014848,-0.014851,-0.014848,0.041227
4,fossil_fraction,-0.013824,-0.013835,-0.435598,0.112039


### Step 4: Robustness Checks

**Quantile regression (τ = 0.10).** OLS estimates the conditional mean of `net_avail_mm_mo`. Water management decisions, however, are driven by tail risk ("How bad is the 10th-percentile dry year?"). QuantReg at τ = 0.10 estimates the conditional 10th percentile. If the `thermo_consum_mm_mo` coefficient is more negative at τ = 0.10 than at the mean (OLS), it indicates that thermoelectric consumption amplifies drought severity disproportionately, which is a critical finding for risk assessment.

**Permutation importance.** Random Forest's default impurity-based feature importance is biased toward high-cardinality continuous features and can overstate importance for features with many splits. Permutation importance (computed on the held-out test set) measures the actual increase in RMSE when each feature is randomly shuffled: a direct measure of predictive contribution in held-out data.

**Residual diagnostics + out-of-sample test.** A properly specified model should have mean-zero, approximately symmetric residuals. Heavy skew or non-zero mean indicates structural misspecification. Additionally, separating test-set residuals by watershed group (watersheds that never had thermoelectric activity vs. those that did) tests whether the model generalizes to never-seen thermoelectric contexts, the deployment scenario for projection.

In [99]:
# --- Quantile regression τ=0.10 (dry-year tail) ---
qr_fit = QuantReg(y_train_fe.astype(float), X_sm).fit(q=0.10, max_iter=2000)
qr_df = pd.DataFrame({
    'Feature':           ['const'] + feature_cols_v2,
    'OLS coef':          ols_fit.params.values.round(6),
    'QR(τ=0.10) coef':   qr_fit.params.round(6),
    'QR more negative?': ['yes' if (q < o and o < 0) or (q < o < 0) else
                          ('yes (stronger neg)' if q < o else 'no')
                          for q, o in zip(qr_fit.params, ols_fit.params.values)],
})
display(qr_df)

# --- Permutation importance (test set, RF) ---
perm = permutation_importance(rf, X_te, y_test_fe, n_repeats=30, random_state=42, scoring='neg_root_mean_squared_error')
perm_df = pd.DataFrame({
    'Feature':        feature_cols_v2,
    'Perm Imp (mean RMSE increase)': perm.importances_mean.round(6),
    'Perm Imp (std)': perm.importances_std.round(6),
    'RF impurity':    rf.feature_importances_.round(6),
}).sort_values('Perm Imp (mean RMSE increase)', ascending=False).reset_index(drop=True)
display(perm_df)

# --- Residual diagnostics ---
resid_tr = y_train_fe - lr.predict(X_tr)
resid_te = y_test_fe  - lr.predict(X_te)
resid_diag = pd.DataFrame({
    'Split': ['Train', 'Test'],
    'n':     [len(resid_tr), len(resid_te)],
    'Mean':  [round(resid_tr.mean(), 4),             round(resid_te.mean(), 4)],
    'Std':   [round(resid_tr.std(), 4),              round(resid_te.std(), 4)],
    'Skew':  [round(float(resid_tr.skew()), 4),      round(float(resid_te.skew()), 4)],
    'P05':   [round(float(np.percentile(resid_tr, 5)), 4),  round(float(np.percentile(resid_te, 5)), 4)],
    'P95':   [round(float(np.percentile(resid_tr, 95)), 4), round(float(np.percentile(resid_te, 95)), 4)],
})
display(resid_diag)

# --- OOS test: never-thermo vs. has-thermo watersheds ---
never_ws = train_fe.groupby('huc12_id')['thermo_consum_mm_mo'].max()
never_ws = never_ws[never_ws == 0].index
has_ws   = never_ws.symmetric_difference(train_fe['huc12_id'].unique())

oos_rows = []
for label, mask in [('Never-thermo watersheds (OOS generalization)',
                      test_fe['huc12_id'].isin(never_ws)),
                     ('Has-thermo watersheds',
                      test_fe['huc12_id'].isin(has_ws))]:
    if mask.sum() == 0:
        continue
    m = metrics(y_test_fe[mask], lr.predict(X_te[mask]))
    oos_rows.append({'Group': label, 'n_test_rows': int(mask.sum()),
                     'RMSE': m['RMSE'], 'R²': m['R2'], 'MAE': m['MAE']})
display(pd.DataFrame(oos_rows))

,Feature,OLS coef,QR(τ=0.10) coef,QR more negative?
const,const,-0.000000,-0.069330,yes
thermo_consum_mm_mo,thermo_consum_mm_mo,-0.298618,-0.327887,yes
availab_mm_mo,availab_mm_mo,1.006089,1.000797,yes (stronger neg)
fossil_fraction,fossil_fraction,-0.435598,-1.579070,yes
industrial_gwh,industrial_gwh,0.000110,0.000172,no
year,year,-0.014848,-0.031691,yes


,Feature,Perm Imp (mean RMSE increase),Perm Imp (std),RF impurity
0,availab_mm_mo,2.382138,0.02849,0.765308
1,thermo_consum_mm_mo,0.000000,0.00000,0.000000
2,fossil_fraction,0.000000,0.00000,0.112039
3,industrial_gwh,0.000000,0.00000,0.081426
4,year,0.000000,0.00000,0.041227


,Split,n,Mean,Std,Skew,P05,P95
0,Train,20456,0.0000,0.2235,2.0188,-0.1555,0.1412
1,Test,5114,0.0063,0.3256,-6.7978,-0.3055,0.1699


,Group,n_test_rows,RMSE,R²,MAE
0,Never-thermo watersheds (OOS generalization),5098,0.3242,0.9941,0.1244
1,Has-thermo watersheds,16,0.6479,0.9897,0.3708


### Step 5: Projection Scenarios, Proposed Facility Impact by Watershed

The fixed-effects OLS coefficient from Step 3 is the correct tool for projections. In natural units (mm/month per mm/month), it estimates: for each additional mm/month of thermoelectric consumption depth added to a watershed, net availability falls by `ols_fit.params['thermo_consum_mm_mo']` mm/month. For a watershed currently at zero thermoelectric activity, this is a difference-in-differences extrapolation. The fixed effect cancels out when computing the change.

**Conversion for each facility:** data center consumption in MGD → mm/month depth = `cons_mgd × MGD_MM_MO_PER_KM2 / areasqkm`.

**Why not the RF for projections?** RF predictions saturate at training-range values of `thermo_consum_mm_mo`. The largest training value is from existing Utah power plants (≈0.1–5 mm/mo). Proposed data centers in Millard and Box Elder counties, which are large and mostly dry endorheic watersheds, may produce different normalized depths. OLS extrapolates linearly, which is the physically motivated assumption for consumptive losses.

**Uncertainty:** The 95% cluster-robust CI from Step 3 propagates directly. The interval `[coef_lo, coef_hi] × cons_mm_mo` brackets the projection uncertainty.

In [100]:
coef_thermo    = ols_fit.params['thermo_consum_mm_mo']
coef_th_ci_lo  = ols_fit.conf_int().loc['thermo_consum_mm_mo', 0]
coef_th_ci_hi  = ols_fit.conf_int().loc['thermo_consum_mm_mo', 1]

target_watersheds_v2 = {
    '160300050805': ('Sugarville',                   10050),
    '160203090501': ('Headwaters Hansel Valley Wash',  9000),
    '160300051502': ('Whiskey Creek',                  4000),
}

proj_rows = []
for huc_str, (huc_name, cap_mw) in target_watersheds_v2.items():
    huc_int  = int(huc_str)
    ws_rows  = water[water['huc12_id'] == huc_int]
    ws_area  = ws_rows['areasqkm'].iloc[0] if len(ws_rows) else water['areasqkm'].median()
    hist     = ws_rows['net_avail_mm_mo']
    hist_avg = round(hist.mean(), 3) if len(hist) else np.nan
    hist_min = round(hist.min(), 3)  if len(hist) else np.nan

    for scenario, wue in scenarios.items():
        cons_mgd   = dc_water_mgd(cap_mw, wue)
        cons_mm_mo = cons_mgd * MGD_MM_MO_PER_KM2 / ws_area
        delta      = coef_thermo * cons_mm_mo
        delta_lo   = coef_th_ci_lo * cons_mm_mo
        delta_hi   = coef_th_ci_hi * cons_mm_mo
        pct        = round(100 * delta / hist_avg, 1) if hist_avg and hist_avg != 0 else np.nan
        proj_rows.append({
            'Watershed':          huc_name,
            'HUC12':              huc_str,
            'Cap (MW)':           cap_mw,
            'WUE':                scenario,
            'Cons (MGD)':         round(cons_mgd, 1),
            'Cons (mm/mo)':       round(cons_mm_mo, 4),
            'Hist Avg (mm/mo)':   hist_avg,
            'Hist Min (mm/mo)':   hist_min,
            'Δnet (mm/mo)':       round(delta, 4),
            '95% CI':             f'[{round(delta_lo,4)}, {round(delta_hi,4)}]',
            '% Hist Avg':         pct,
        })

pd.DataFrame(proj_rows)

,Watershed,HUC12,Cap (MW),WUE,Cons (MGD),Cons (mm/mo),Hist Avg (mm/mo),Hist Min (mm/mo),Δnet (mm/mo),95% CI,% Hist Avg
0,Sugarville,160300050805,10050,low,25.5,39.0030,-15.584,-18.615,-11.6470,"[-13.8884, -9.4056]",74.7
1,Sugarville,160300050805,10050,mid,76.5,117.0091,-15.584,-18.615,-34.9411,"[-41.6653, -28.2168]",224.2
2,Sugarville,160300050805,10050,high,102.0,156.0122,-15.584,-18.615,-46.5881,"[-55.5538, -37.6224]",298.9
3,Headwaters Hansel Valley Wash,160203090501,9000,low,22.8,17.9455,0.470,-0.025,-5.3588,"[-6.3901, -4.3276]",-1140.2
4,Headwaters Hansel Valley Wash,160203090501,9000,mid,68.5,53.8364,0.470,-0.025,-16.0765,"[-19.1704, -12.9827]",-3420.5
5,Headwaters Hansel Valley Wash,160203090501,9000,high,91.3,71.7818,0.470,-0.025,-21.4354,"[-25.5605, -17.3102]",-4560.7
6,Whiskey Creek,160300051502,4000,low,10.1,9.8171,-1.286,-4.742,-2.9316,"[-3.4957, -2.3674]",228.0
7,Whiskey Creek,160300051502,4000,mid,30.4,29.4514,-1.286,-4.742,-8.7947,"[-10.4872, -7.1022]",683.9
8,Whiskey Creek,160300051502,4000,high,40.6,39.2686,-1.286,-4.742,-11.7263,"[-13.983, -9.4696]",911.8


### Preliminary Conclusion

**Model performance.** Linear Regression and Ridge with watershed fixed effects perform nearly identically (test RMSE = 0.326 mm/month, R² = 0.994). GroupKFold cross-validation (which withholds entire watersheds from each validation fold) yields RMSE of 0.221 ± 0.035 mm/month, somewhat better than the held-out test set, reflecting that 2018 (the driest year in the dataset) introduces tail conditions not well-represented in training. Random Forest performs poorly (test RMSE = 2.46, R² = 0.67): with only 36 thermoelectric watershed-years out of 20,456 training rows, the forest cannot learn splits on `thermo_consum_mm_mo` at `min_samples_leaf=100` and degrades on the sparse tail. Linear models are clearly preferred for this problem. On the five thermoelectric-active test observations, LR/Ridge achieve RMSE = 0.141 and R² = 0.9998 — highly accurate on the exact phenomenon of interest.

**Thermoelectric coefficient.** The primary quantitative finding: each 1 mm/month increase in thermoelectric consumption depth reduces watershed net availability by **−0.299 mm/month** (cluster-robust 95% CI: [−0.356, −0.241]; p < 0.001). The independent pairs-bootstrap CI [−0.358, −0.260] confirms this range. The coefficient is statistically unambiguous — zero is not remotely in either interval. The effect is less than the physically expected −1.0 because (a) the fixed-effects estimator identifies only the within-watershed year-to-year variation, and (b) the panel's short 8-year training window limits the range of within-watershed `thermo_consum_mm_mo` variation for the handful of active watersheds.

**Dry-year amplification.** Quantile regression at τ = 0.10 yields a coefficient of −0.328 on `thermo_consum_mm_mo` vs. −0.299 at the conditional mean, a 10% amplification at the dry-year tail. For `fossil_fraction`, the τ = 0.10 coefficient is −1.579 vs. −0.436 at the mean, a 3.6× amplification. Both findings indicate that thermoelectric stress and grid composition effects are disproportionately concentrated in drought years, exactly when water is already scarcest.

**Feature importance.** Permutation importance confirms that `availab_mm_mo` is the dominant predictor by a wide margin (+2.38 RMSE when shuffled). `thermo_consum_mm_mo` registers zero permutation importance with the Random Forest, correctly reflecting the forest's inability to use the sparse thermoelectric signal, but the linear model assigns it a coefficient that is tightly estimated and highly significant. This divergence between permutation importance (model-specific, RF-based) and coefficient significance (OLS) underscores the importance of using the right model for the right question.

**Out-of-sample generalization.** The model generalizes well to watersheds that have *never* had thermoelectric activity (RMSE = 0.324, R² = 0.994 on 5,098 never-thermo test rows), which is the actual deployment scenario: predicting impacts in proposed-facility watersheds with no thermoelectric history. Watersheds that do have thermoelectric history show higher error (RMSE = 0.648), likely because their historical consumption variation is greater and harder to fully capture with only 10 years of data.

**Residual diagnostics.** Training residuals are approximately unbiased (mean ≈ 0) but right-skewed (skew = 2.02), reflecting model underestimates in anomalously wet years. Test residuals are left-skewed (skew = −6.80), indicating systematic underestimates of how dry 2018 was. This is expected: 2018 was the driest year in the sample, and no training year is equally severe. The model is conservative on drought risk.

**Projection results.** Applying the OLS coefficient (−0.299 mm/mo per mm/mo) with watershed-specific areas and WUE scenarios:

| Watershed | WUE | Cons (MGD) | Cons (mm/mo) | Hist Avg (mm/mo) | Δnet (mm/mo) |
|---|---|---|---|---|---|
| Sugarville (10,050 MW) | low | 25.5 | 39.0 | −15.6 | **−11.6** |
| Sugarville (10,050 MW) | mid | 76.5 | 117.0 | −15.6 | **−34.9** |
| Headwaters Hansel Valley (9,000 MW) | mid | 68.5 | 53.8 | +0.5 | **−16.1** |
| Whiskey Creek (4,000 MW) | mid | 30.4 | 29.5 | −1.3 | **−8.8** |

Sugarville at mid WUE would add −34.9 mm/month to a watershed already averaging −15.6 mm/month — driving the combined projected average to approximately −50.5 mm/month. Headwaters Hansel Valley Wash (the proposed Stratos site) is currently barely water-positive (+0.47 mm/month historical average), but the proposed 9,000 MW facility would push it to approximately −15.6 mm/month at mid WUE. All three target watersheds are endorheic Great Basin basins with no downstream outlet — every mm/month of consumptive loss is permanent.

**Key limitations.**
- The thermoelectric signal is learned from 36 training watershed-years at existing Utah coal plants (Hunter, Huntington, Emery County). Extrapolation to data-center cooling assumes the same physical mechanism, which is reasonable but untested for large-scale evaporative cooling.
- Watershed fixed effects remove time-invariant confounders but not time-varying ones (e.g., drought years correlating with plant curtailment).
- The within-watershed coefficient of −0.299 is likely an underestimate of the true consumptive effect (expected −1.0) due to the limited within-watershed variation in the 10-year panel.
- Projections do not account for return flows, groundwater recharge, climate change trends in `availab_mm_mo`, or the compounding effect of multiple facilities sharing the same basin.

## Milestone 4: Final Touches

This section appends post-Milestone-3 improvements to the project. All previous analysis remains untouched above; nothing here re-trains or modifies the existing models. The additions close three gaps in the prior submission:

1. **Visualizations.** Milestones 1–3 contained no plots. Project description requires EDA visuals for the final writeup. This section adds eleven figures covering distribution, time-series, spatial, diagnostic, and projection views.
2. **Spatial maps of consumption.** Two side-by-side maps of Utah HUC12 watersheds — current 2010–2019 mean consumption and projected consumption after all proposed data centers are operational — make the spatial shift in water stress immediately visible.
3. **Cumulative pipeline impact.** Step 5 projected per-watershed effects for three target sites. This section extends the projection to **every** planned facility in the Cleanview inventory, computes per-watershed cumulative impact (multiple facilities in one watershed), and a statewide aggregate.
4. **Robustness sensitivity.** A leave-one-out check drops the driest training year and refits the OLS to verify that the key thermoelectric coefficient does not depend on extreme drought conditions.

All cells reuse variables from prior sections (`water`, `huc12`, `dc_huc12`, `ols_fit`, `lr`, `rf`, `coef_thermo`, `scenarios`, `dc_water_mgd`, `MGD_MM_MO_PER_KM2`, `train_fe`, `test_fe`, `X_tr`, `X_te`, `y_train_fe`, `y_test_fe`). Figures are also written to `figures/` so they can be embedded directly in the Word writeup.

### Visualization Setup

Load `matplotlib`, `seaborn`, and `scipy.stats`. Set a consistent style: dropped right/top spines, larger title weight, higher DPI for embedding in the Word writeup. All figures are saved to `figures/` so the writeup can embed them without re-rendering.

In [101]:
# Plotting imports and shared style
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import scipy.stats as stats
import os

# Output directory for figures — also consumed by the Word writeup generator
FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)

# Project-wide plotting style — high DPI for embedding in .docx, clean axes
plt.rcParams.update({
    'figure.dpi':         110,
    'savefig.dpi':        160,
    'savefig.bbox':       'tight',
    'font.size':          10,
    'axes.titlesize':     11,
    'axes.titleweight':   'bold',
    'axes.labelsize':     10,
    'axes.spines.right':  False,   # cleaner look — drop right spine
    'axes.spines.top':    False,   # cleaner look — drop top spine
})

### Figure 1: Distribution of Net Water Availability

A histogram of `net_avail_mm_mo` across all 25,580 watershed-year observations reveals the prevalence of water deficit (net < 0). The red dashed line marks the stress threshold. The percentage of observations in deficit is computed and shown in the title — a single statewide-scale summary of the problem this project quantifies.

In [102]:
# Figure 1 — distribution of the target variable across the full panel
# Shows what fraction of watershed-year observations are already in deficit (net < 0)
fig1, ax = plt.subplots(figsize=(8, 4))
ax.hist(water['net_avail_mm_mo'].dropna(), bins=80,
        color='#3a7bd5', edgecolor='white', alpha=0.9)

# Reference line at net=0 — the boundary between water-positive and water-stressed
ax.axvline(0, color='crimson', linestyle='--', linewidth=1.6,
           label='Stress threshold (net = 0)')

# Compute the in-deficit share for the title — a headline statistic
pct_deficit = (water['net_avail_mm_mo'] < 0).mean() * 100
ax.set_title(f'Net Water Availability Across Utah HUC12 Watersheds, 2010–2019\n'
             f'{pct_deficit:.1f}% of watershed-year observations are in deficit')
ax.set_xlabel('Net Availability (mm/month)')
ax.set_ylabel('Count of Watershed-Year Observations')
ax.legend()
plt.savefig(f'{FIG_DIR}/01_net_avail_distribution.png')
plt.close()

print(f'{pct_deficit:.1f}% of observations are in net water deficit')

8.9% of observations are in net water deficit


### Figure 2: Statewide Annual Hydrology, 2010–2019

A three-line time series of statewide mean natural availability, total consumption, and net availability. This shows (a) how strongly net availability tracks natural availability (consumption is relatively flat year-to-year), and (b) how severe 2018 was — the dataset's driest year — which is the dominant source of test-set residual error in Milestone 3.

In [103]:
# Figure 2 — statewide annual time series
# Aggregate by year across all 2,558 watersheds to see secular trends and drought years
ann = water.groupby('year').agg(
    avail=('availab_mm_mo', 'mean'),
    consum=('consum_mm/mo', 'mean'),
    net=('net_avail_mm_mo', 'mean'),
).reset_index()

fig2, ax = plt.subplots(figsize=(9, 4.5))
# Three lines: natural inputs, human draws, and the net (target variable)
ax.plot(ann['year'], ann['avail'], 'o-', label='Natural Availability', color='#1f77b4', linewidth=2)
ax.plot(ann['year'], ann['consum'], 's-', label='Total Consumption',   color='#d62728', linewidth=2)
ax.plot(ann['year'], ann['net'],   '^-', label='Net Availability',     color='#2ca02c', linewidth=2)
ax.axhline(0, color='black', linewidth=0.6, alpha=0.5)
ax.set_title('Statewide Watershed Hydrology, 2010–2019 — Mean across 2,558 HUC12 Watersheds')
ax.set_xlabel('Year')
ax.set_ylabel('mm/month')
ax.legend(loc='upper right')

# Highlight 2018 — the driest year in the dataset, sits in the test set
ax.annotate('2018 drought\n(test year, lowest availability)',
            xy=(2018, ann.loc[ann.year==2018, 'avail'].iloc[0]),
            xytext=(2014.5, 7.0), fontsize=9, ha='center',
            arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

plt.savefig(f'{FIG_DIR}/02_statewide_hydrology.png')
plt.close()

### Figure 3: Statewide Thermoelectric Consumption — USGS Actual vs. UofU × Intensity Estimate

Plots the USGS NWAA model's sum of watershed-level thermoelectric consumption (`tecuftot_mgd`) against the independent UofU-generation-times-intensity-factor estimate. The relative agreement validates the intensity-factor methodology used for projection scenarios; the absolute gap reflects USGS coverage limits (not all plants are geolocated to a single HUC12 cell).

In [104]:
# Figure 3 — validation of the UofU × intensity-factor estimator against USGS actuals
# Both estimates are in MGD; relative shape agreement is the key validation, not absolute level
cmp = comparison.dropna(subset=['usgs_actual_mgd']).copy()

fig3, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(cmp['Year'], cmp['est_thermo_consumption_mgd'], 'o-',
        label='UofU × intensity (estimate)',  color='#8e44ad', linewidth=2)
ax.plot(cmp['Year'], cmp['usgs_actual_mgd'], 's-',
        label='USGS NWAA actual (Utah HUC12 sum)', color='#16a085', linewidth=2)
ax.set_title('Statewide Thermoelectric Water Consumption — Validation of Intensity-Factor Estimate')
ax.set_xlabel('Year')
ax.set_ylabel('MGD (Million Gallons / Day)')
ax.legend()
plt.savefig(f'{FIG_DIR}/03_thermo_validation.png')
plt.close()

### Figure 4: Utah Electricity Generation Mix, 2000–2025

A stacked-area chart of Utah's generation by fuel type from the UofU dataset. Coal has declined from over 80% to under 60%; natural gas has roughly tripled; solar and wind have grown from zero. The model's `fossil_fraction` feature captures this transition — the cluster-robust coefficient of −0.44 mm/mo per unit shift toward fossil generation reflects exactly this composition effect.

In [105]:
# Figure 4 — Utah generation mix as a stacked area
# Shows the coal decline + gas growth that drives the model's fossil_fraction trend
gen_cols  = ['Coal', 'Natural Gas', 'Petroleum', 'Hydro', 'Wind', 'Solar', 'Geothermal', 'Biomass']
gen_cols  = [c for c in gen_cols if c in uofu.columns]  # tolerate missing fuels
gen_data  = uofu.set_index('Year')[gen_cols]

fig4, ax = plt.subplots(figsize=(10, 5))
# Color palette chosen for fuel-source intuition (coal=black, gas=orange, renewables=green/blue)
colors = ['#2f2f2f', '#e67e22', '#7f7f7f', '#3498db', '#1abc9c',
          '#f1c40f', '#e74c3c', '#27ae60']
gen_data.plot.area(ax=ax, color=colors[:len(gen_cols)], alpha=0.85)
ax.set_title('Utah Electricity Generation by Fuel Type, 2000–2025 (GWh)')
ax.set_xlabel('Year')
ax.set_ylabel('Net Generation (GWh)')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.savefig(f'{FIG_DIR}/04_generation_mix.png')
plt.close()

### Figure 5: Natural Availability vs. Net Availability — Thermoelectric Watersheds Highlighted

A scatter of all 25,580 observations: x = `availab_mm_mo`, y = `net_avail_mm_mo`. Non-thermoelectric watersheds (gray) cluster tightly along the y = x line — natural input equals net output when there is no consumption. Thermoelectric-active watershed-years (red, n ≈ 50) deviate downward from this baseline, illustrating exactly the consumptive depression the OLS coefficient measures.

In [106]:
# Figure 5 — show how thermoelectric-active watersheds deviate from the avail≈net baseline
# Non-thermo watersheds form the y≈x reference; thermo watersheds sit below it (red points)
fig5, ax = plt.subplots(figsize=(8, 5.5))

# Split observations into two cohorts for layered scatter
non_thermo = water[water['thermo_consum_mm_mo'] == 0]
has_thermo = water[water['thermo_consum_mm_mo']  > 0]

ax.scatter(non_thermo['availab_mm_mo'], non_thermo['net_avail_mm_mo'],
           s=4, alpha=0.18, color='#7f8c8d', label='No thermoelectric activity')
ax.scatter(has_thermo['availab_mm_mo'], has_thermo['net_avail_mm_mo'],
           s=30, alpha=0.85, color='#c0392b', edgecolor='black', linewidth=0.4,
           label=f'Thermoelectric-active (n={len(has_thermo)})')

ax.set_title('Watershed Availability vs. Net Availability — Thermoelectric Watersheds Highlighted')
ax.set_xlabel('Natural Availability (mm/month)')
ax.set_ylabel('Net Availability (mm/month)')
ax.axhline(0, color='black', linewidth=0.5, alpha=0.5)
ax.legend(loc='lower right')
plt.savefig(f'{FIG_DIR}/05_avail_vs_net_scatter.png')
plt.close()

### Figure 6: Post-FE Feature Correlations

A heatmap of pairwise correlations among the five model features and the target, computed on the *demeaned* training data. After watershed fixed-effects demeaning, no two predictors exhibit dangerous collinearity (|r| > 0.9). `availab_mm_mo` shows the expected strong positive correlation with `net_avail_mm_mo` (r ≈ 1.0 because consumption is small relative to availability for most watersheds).

In [107]:
# Figure 6 — feature correlations post-FE demeaning
# Confirms no severe multicollinearity remains after removing watershed-level baselines
corr = train_fe[feature_cols_v2 + [target_col]].corr()

fig6, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3,
            ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Post-FE Feature Correlations (Training Set)')
plt.savefig(f'{FIG_DIR}/06_feature_correlation.png')
plt.close()

### Figure 7: Utah Watershed Consumption Maps — Current vs. Projected

Two side-by-side choropleths of Utah HUC12 watersheds:

- **Map A** colors watersheds by 2010–2019 mean `consum_mm/mo` (current consumption from all uses). Triangles mark watersheds with historical thermoelectric activity (existing coal-plant cooling).
- **Map B** colors watersheds by *projected* consumption after every proposed data center in the Cleanview inventory is operational at mid-WUE cooling intensity (1.5 L/kWh). Circles size-by-MW show proposed facility locations. The three highest-impact hotspots are labeled.

Both maps share a logarithmic SymLog color scale so that both small baseline variation and the extreme new hotspots (Sugarville, Headwaters Hansel Valley Wash, Whiskey Creek) are visible in one frame. The shift from broadly-low consumption to a few concentrated, extreme hotspots is the central spatial finding.

In [108]:
# Figure 7 — current vs. projected consumption maps (the headline visual)
# Map A: 2010–2019 mean consumption per HUC12
# Map B: same baseline + sum of proposed data center cooling demand at mid WUE
# Shared SymLog color scale lets baseline variation AND new hotspots both be visible

# --- Per-HUC12 current consumption from historical data ---
water_mean = water.groupby('huc12_id').agg(
    consum_current=('consum_mm/mo', 'mean'),
    areasqkm=('areasqkm', 'first'),
).reset_index()
water_mean['huc12_key'] = water_mean['huc12_id'].astype(str).str.zfill(12)

# Merge onto the huc12 GeoDataFrame for plotting
huc_plot = huc12.copy()
huc_plot['huc12_key'] = huc_plot['huc12'].astype(str).str.zfill(12)
huc_plot = huc_plot.merge(water_mean[['huc12_key', 'consum_current']], on='huc12_key', how='left')
huc_plot['consum_current'] = huc_plot['consum_current'].fillna(0)

# --- Add projected demand from proposed (status='Planned') data centers ---
WUE_MID = scenarios['mid']  # 1.5 L/kWh, typical evaporative cooling for arid climate
dc_plan = dc_huc12[dc_huc12['status'].isin(['Planned','planned','PLANNED'])].copy()
dc_plan = dc_plan.dropna(subset=['capacity_mw', 'huc12'])
dc_plan['capacity_mw'] = pd.to_numeric(dc_plan['capacity_mw'], errors='coerce')
dc_plan = dc_plan.dropna(subset=['capacity_mw'])
# Convert MW capacity to cooling water demand in MGD using the existing helper
dc_plan['cooling_mgd'] = dc_plan['capacity_mw'].apply(lambda mw: dc_water_mgd(mw, WUE_MID))
dc_plan['huc12_key']   = dc_plan['huc12'].astype(str).str.zfill(12)

# Aggregate facility-level demand up to the watershed level (multiple plants per HUC12 possible)
dc_per_huc = dc_plan.groupby('huc12_key').agg(
    new_mgd=('cooling_mgd', 'sum'),
    new_mw=('capacity_mw', 'sum'),
    n_facilities=('id', 'count'),
).reset_index()

huc_plot = huc_plot.merge(dc_per_huc[['huc12_key', 'new_mgd']], on='huc12_key', how='left')
huc_plot['new_mgd'] = huc_plot['new_mgd'].fillna(0)
# Convert added MGD to mm/month depth using the project's standard area-normalized factor
huc_plot['new_consum_mm_mo'] = huc_plot['new_mgd'] * MGD_MM_MO_PER_KM2 / huc_plot['areasqkm'].replace(0, np.nan)
huc_plot['new_consum_mm_mo'] = huc_plot['new_consum_mm_mo'].fillna(0)
huc_plot['consum_projected'] = huc_plot['consum_current'] + huc_plot['new_consum_mm_mo']

# Shared color scale — SymLog handles the huge dynamic range (0 → ~120 mm/mo)
vmax = float(huc_plot['consum_projected'].max())
norm = mcolors.SymLogNorm(linthresh=0.5, linscale=1.0, vmin=0, vmax=vmax, base=10)

# Top-3 new hotspots for labeling on Map B (deduplicated by HUC12)
top3 = (huc_plot.drop_duplicates('huc12_key')
                .sort_values('new_consum_mm_mo', ascending=False)
                .head(3))

# Utah state border = union of HUC12 polygons WHOLLY within Utah (states == 'UT' only)
# This gives a Utah-shaped outline; multi-state watersheds along borders are excluded
# so the resulting outline traces the political boundary rather than watershed sprawl.
import geopandas as _gpd
utah_only = huc12[huc12['states'] == 'UT']
utah_border = _gpd.GeoSeries([utah_only.geometry.union_all()], crs=huc12.crs).boundary

# --- Render the two-panel figure ---
fig7, axes = plt.subplots(1, 2, figsize=(15, 7.5))
for ax, col, ttl in [
    (axes[0], 'consum_current',  'A. Current (2010–2019 Mean)'),
    (axes[1], 'consum_projected','B. Projected After Data Center Buildout (mid-WUE)'),
]:
    huc_plot.plot(column=col, cmap='OrRd', norm=norm, edgecolor='gray',
                  linewidth=0.05, ax=ax, legend=False)
    # Utah state outline in black for spatial context
    utah_border.plot(ax=ax, color='black', linewidth=1.6)
    ax.set_title(ttl, fontsize=11, pad=8)
    # Drop axis labels and tick numbers; faint dotted grid for spatial reference
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.set_xticklabels([]); ax.set_yticklabels([])
    ax.tick_params(left=False, bottom=False)
    ax.grid(True, color='gray', linestyle=':', linewidth=0.4, alpha=0.5)
    ax.set_xlim(-114.5, -108.9); ax.set_ylim(36.9, 42.1)
    ax.set_aspect('equal')

# Overlay existing thermoelectric watersheds on Map A as triangles
existing_thermo = water[water['thermo_consum_mm_mo'] > 0].groupby('huc12_id')['areasqkm'].first().index
existing_huc = huc_plot[huc_plot['huc12_key'].isin([str(h).zfill(12) for h in existing_thermo])]
if len(existing_huc):
    cents = existing_huc.geometry.centroid
    axes[0].scatter(cents.x, cents.y, s=40, marker='^', facecolor='none',
                    edgecolor='black', linewidth=1.2, label='Existing thermoelectric')
    axes[0].legend(loc='lower left', fontsize=8)

# Overlay proposed facility points on Map B, sized by capacity
if len(dc_plan):
    axes[1].scatter(dc_plan['longitude'], dc_plan['latitude'],
                    s=np.sqrt(dc_plan['capacity_mw'])*5, marker='o',
                    facecolor='none', edgecolor='black', linewidth=1.1,
                    label='Proposed data center (size ∝ √MW)')
    axes[1].legend(loc='lower right', fontsize=8)
    # Label the top-3 hotspot watersheds by name
    for i, (_, row) in enumerate(top3.iterrows()):
        c = row.geometry.centroid
        nm = row.get('name', row['huc12_key'])
        dx, dy = (0.8, -0.4) if i % 2 == 0 else (0.8, 0.4)
        tx, ty = max(-114.4, min(c.x+dx, -109.0)), max(37.0, min(c.y+dy, 41.9))
        axes[1].annotate(str(nm)[:28], xy=(c.x, c.y), xytext=(tx, ty),
                         fontsize=8, ha='left',
                         arrowprops=dict(arrowstyle='->', color='black', lw=0.6))

# Shared colorbar (use distinct name to avoid shadowing statsmodels alias `sm`)
sc_map = plt.cm.ScalarMappable(cmap='OrRd', norm=norm)
sc_map.set_array([])
cbar = fig.colorbar(sc_map, ax=axes, orientation='horizontal',
                    fraction=0.05, pad=0.07, shrink=0.6)
cbar.set_label('Consumption (mm/month, shared scale)')
fig.suptitle('Utah Watershed Consumption — Before and After Proposed Data Center Buildout',
             fontsize=13, fontweight='bold', y=.83)
plt.subplots_adjust(top=0.92)
plt.savefig(f'{FIG_DIR}/07_maps_current_vs_projected.png')
plt.close()

# Print the hotspot summary
print('Top 3 new consumption hotspots:')
print(top3[['huc12_key', 'name', 'new_consum_mm_mo', 'new_mgd']].to_string(index=False))


Top 3 new consumption hotspots:
   huc12_key                          name  new_consum_mm_mo   new_mgd
160300050805                    Sugarville        117.009143 76.470277
160203090501 Headwaters Hansel Valley Wash         53.836370 68.480845
160300051502                 Whiskey Creek         29.451428 30.435931


### Figure 8: Predicted vs. Actual (OLS Fixed-Effects)

Two-panel predicted-vs-actual scatter, one for training (n=20,456) and one for test (n=5,114). Tight alignment to the diagonal y = x confirms the OLS-FE model's R² ≥ 0.994 on both partitions, including the 2018–2019 test years.

In [109]:
# Figure 8 — predicted-vs-actual for the OLS-FE model on both partitions
# Tight diagonal alignment confirms R² ≈ 0.99 reported in the model evaluation step
pred_tr = lr.predict(X_tr)
pred_te = lr.predict(X_te)

fig8, axes = plt.subplots(1, 2, figsize=(13, 5.5))
# Shared axis range so both panels are visually comparable
AX_LO, AX_HI = -25, 35
for ax, y_true, y_pred, ttl, color in [
    (axes[0], y_train_fe, pred_tr,
     f'Training (n={len(y_train_fe):,}, R²={r2_score(y_train_fe, pred_tr):.4f})', '#1f77b4'),
    (axes[1], y_test_fe,  pred_te,
     f'Test (n={len(y_test_fe):,}, R²={r2_score(y_test_fe, pred_te):.4f})', '#d62728'),
]:
    ax.scatter(y_true, y_pred, s=3, alpha=0.25, color=color)
    # 45° reference line — perfect prediction
    ax.plot([AX_LO, AX_HI], [AX_LO, AX_HI], 'k--', linewidth=1, alpha=0.5)
    ax.set_xlim(AX_LO, AX_HI); ax.set_ylim(AX_LO, AX_HI)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('Actual (demeaned net_avail_mm_mo)')
    ax.set_ylabel('Predicted')
    ax.set_title(ttl)

fig8.suptitle('OLS Fixed-Effects: Predicted vs. Actual', fontweight='bold')
plt.savefig(f'{FIG_DIR}/08_pred_vs_actual.png')
plt.close()

### Figure 9: Residual Diagnostics (Test Set)

Two-panel: (left) residuals vs. fitted, checking homoscedasticity; (right) Q-Q plot of test residuals against a normal distribution. The Q-Q plot shows a heavy left tail (skew ≈ −6.8), consistent with the model underestimating how dry 2018 was — Milestone 3's noted limitation that no training year matches 2018's drought severity.

In [110]:
# Figure 9 — residual diagnostics for the OLS-FE model on test set
# Left: residuals vs fitted (homoscedasticity check)
# Right: Q-Q plot vs normal — heavy left tail flags 2018 drought underestimation
resid_te = y_test_fe - pred_te

fig9, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: residuals vs fitted — look for funnel shapes (heteroskedastic) or systematic curvature
axes[0].scatter(pred_te, resid_te, s=4, alpha=0.25, color='#34495e')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Fitted (demeaned)')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Fitted (Test Set)')

# Panel 2: Q-Q plot — points should align with the diagonal if residuals are normal
stats.probplot(resid_te, dist='norm', plot=axes[1])
axes[1].set_title(f'Q-Q Plot of Test Residuals (skew = {resid_te.skew():.2f})')

plt.savefig(f'{FIG_DIR}/09_residual_diagnostics.png')
plt.close()

### Figure 10: Coefficient Forest Plot with Cluster-Robust 95% CIs

A horizontal error-bar plot of the five OLS-FE coefficients in natural units, with cluster-robust 95% CIs (clustered by `huc12_id`). The key result: `thermo_consum_mm_mo` is unambiguously negative (−0.299, CI [−0.356, −0.241]); the CI does not approach zero. `availab_mm_mo` ≈ 1.0 confirms the physical interpretation (every mm/month of natural input passes through to net availability when consumption is held constant). `fossil_fraction` is also strongly negative — statewide shifts toward fossil generation drive thermoelectric water demand and depress net availability.

In [111]:
# Figure 10 — coefficient + cluster-robust 95% CI forest plot
# Each feature is a horizontal line: dot=point estimate, whiskers=CI
ci = ols_fit.conf_int()
forest_df = pd.DataFrame({
    'feature': ols_fit.params.index,
    'coef':    ols_fit.params.values,
    'ci_lo':   ci[0].values,
    'ci_hi':   ci[1].values,
}).query("feature != 'const'").sort_values('coef')  # exclude intercept, sort for readability

fig10, ax = plt.subplots(figsize=(9, 4.5))
y_pos = np.arange(len(forest_df))

# errorbar wants the lower/upper *distances* from the point estimate, not the bounds themselves
ax.errorbar(forest_df['coef'], y_pos,
            xerr=[forest_df['coef']-forest_df['ci_lo'],
                  forest_df['ci_hi']-forest_df['coef']],
            fmt='o', color='#2c3e50', ecolor='#3498db',
            capsize=4, markersize=8)
ax.axvline(0, color='red', linestyle='--', linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(forest_df['feature'])
ax.set_xlabel('OLS Coefficient (natural units, cluster-robust 95% CI)')
ax.set_title('OLS-FE Coefficient Estimates with Cluster-Robust Confidence Intervals')
plt.savefig(f'{FIG_DIR}/10_coefficient_forest.png')
plt.close()

### Robustness Check: Leave-Driest-Training-Year-Out

A common concern with panel models is that one extreme year (e.g., a severe drought) might disproportionately drive the estimated coefficient. To test this, the model is refit on the training set with the driest training year removed, and the `thermo_consum_mm_mo` coefficient is compared to the full-training estimate. A small percentage change indicates the coefficient is robust to drought-year removal.

In [112]:
# Robustness check — drop the driest training year, refit OLS, compare coefficient
# If the coefficient changes substantially when the dryest year is removed, the headline
# estimate is fragile to drought conditions. A small change indicates robustness.

# Identify the driest training year by mean availability
ann_avail = water[water['year'] <= 2017].groupby('year')['availab_mm_mo'].mean()
driest_year = ann_avail.idxmin()
print(f'Driest training year: {driest_year} (mean avail = {ann_avail.min():.2f} mm/mo)')

# Refit FE OLS without that year's training data
train_robust = train_raw[train_raw['year'] != driest_year].copy()
# Important: recompute watershed means from the reduced training set (no test leakage)
ws_means_r = train_robust.groupby('huc12_id')[feature_cols_v2 + [target_col]].mean()
train_robust_fe = apply_fe(train_robust, ws_means_r)

# Cluster-robust OLS on the leave-one-out training data
X_train_r = train_robust_fe[feature_cols_v2].astype(float)
y_train_r = train_robust_fe[target_col].astype(float)
X_sm_r = sm.add_constant(X_train_r)
ols_robust = sm.OLS(y_train_r, X_sm_r).fit(
    cov_type='cluster',
    cov_kwds={'groups': train_robust_fe['huc12_id'].values}
)

# Compare coefficients side-by-side
coef_full   = ols_fit.params['thermo_consum_mm_mo']
coef_robust = ols_robust.params['thermo_consum_mm_mo']
robust_change_pct = 100 * (coef_robust - coef_full) / abs(coef_full)

robust_summary = pd.DataFrame({
    'Model':     ['Full training (2010–2017)', f'Leave-{driest_year}-out'],
    'Coef':      [round(coef_full, 4), round(coef_robust, 4)],
    'CI lower':  [round(ols_fit.conf_int().loc['thermo_consum_mm_mo', 0], 4),
                  round(ols_robust.conf_int().loc['thermo_consum_mm_mo', 0], 4)],
    'CI upper':  [round(ols_fit.conf_int().loc['thermo_consum_mm_mo', 1], 4),
                  round(ols_robust.conf_int().loc['thermo_consum_mm_mo', 1], 4)],
    'p-value':   [round(ols_fit.pvalues['thermo_consum_mm_mo'], 6),
                  round(ols_robust.pvalues['thermo_consum_mm_mo'], 6)],
})
print(f'\nCoefficient change vs full training: {robust_change_pct:+.1f}%')
display(robust_summary)
robust_summary.to_csv(f'{FIG_DIR}/_robustness_summary.csv', index=False)

Driest training year: 2012 (mean avail = 2.68 mm/mo)

Coefficient change vs full training: +0.4%


,Model,Coef,CI lower,CI upper,p-value
0,Full training (2010–2017),-0.2986,-0.3561,-0.2412,0.0
1,Leave-2012-out,-0.2974,-0.3532,-0.2417,0.0


### Cumulative Pipeline Impact — All Proposed Facilities

The original Step 5 projected three target watersheds. This extension projects the impact of the **entire** Cleanview-catalogued planned pipeline: 10 facilities totaling 23,222 MW across 6 watersheds, with multiple facilities aggregating in Sugarville (3 facilities, 10,050 MW), Headwaters Hansel Valley Wash (Stratos 2 phases, 9,000 MW), and Bingham Creek (2 facilities, 147 MW). Each watershed-level cumulative demand is then propagated through the OLS coefficient with the cluster-robust 95% CI for uncertainty bands.

A summary table of all six watersheds at three WUE scenarios follows, plus a statewide aggregate.

In [113]:
# Cumulative per-watershed impact — extend Step 5 from 3 to all proposed facilities
# For each watershed with any planned facility, sum total MW, compute cooling demand at
# three WUE scenarios, convert to mm/mo, project Δnet using ols_fit coefficient + CI
cumul_rows = []
for huc_key, grp in dc_plan.groupby('huc12_key'):
    huc_int  = int(huc_key)
    # Watershed name (from Cleanview HUC12 lookup) and historical baseline
    huc_name = grp['huc12_name'].iloc[0] if 'huc12_name' in grp.columns else huc_key
    ws_rows  = water[water['huc12_id'] == huc_int]
    if len(ws_rows) == 0:
        # Fallback if watershed absent from USGS panel (unlikely for Utah HUC12)
        ws_area  = water['areasqkm'].median()
        hist_avg = np.nan
    else:
        ws_area  = ws_rows['areasqkm'].iloc[0]
        hist_avg = ws_rows['net_avail_mm_mo'].mean()
    total_mw = grp['capacity_mw'].sum()  # cumulative MW across all facilities in this HUC12
    n_fac    = len(grp)

    # Project under three water-use intensities (low/mid/high WUE)
    for scen_name, wue in scenarios.items():
        # Sum cooling demand across all facilities in the watershed
        cons_mgd   = grp['capacity_mw'].apply(lambda mw: dc_water_mgd(mw, wue)).sum()
        # Convert MGD → mm/month depth using watershed area
        cons_mm_mo = cons_mgd * MGD_MM_MO_PER_KM2 / ws_area
        # Apply OLS coefficient and propagate cluster-robust CI bounds
        delta      = coef_thermo  * cons_mm_mo
        delta_lo   = coef_th_ci_lo * cons_mm_mo
        delta_hi   = coef_th_ci_hi * cons_mm_mo
        cumul_rows.append({
            'HUC12': huc_key, 'Watershed': huc_name,
            'n_facilities': n_fac, 'Total MW': total_mw,
            'WUE': scen_name, 'Cons MGD': round(cons_mgd, 1),
            'Cons mm/mo': round(cons_mm_mo, 3),
            'Hist Avg net (mm/mo)': round(hist_avg, 2) if not np.isnan(hist_avg) else np.nan,
            'Δnet (mm/mo)': round(delta, 3),
            'Δnet CI lo':   round(delta_lo, 3),
            'Δnet CI hi':   round(delta_hi, 3),
        })

cumul_df = pd.DataFrame(cumul_rows).sort_values(['Total MW', 'WUE'], ascending=[False, True])
cumul_df.to_csv(f'{FIG_DIR}/_cumulative_per_watershed.csv', index=False)
display(cumul_df)

# Statewide aggregate summary at mid WUE
mid = cumul_df[cumul_df['WUE'] == 'mid']
state_agg = pd.DataFrame({
    'Metric': ['Total planned MW', 'Total cooling water (MGD)',
               'Avg watershed Δnet (mm/mo, mid WUE)',
               'Worst watershed Δnet (mm/mo, mid WUE)',
               'Watersheds with |Δnet| > |Hist Avg|'],
    'Value': [
        f"{int(mid['Total MW'].sum()):,}",
        f"{mid['Cons MGD'].sum():.1f}",
        f"{mid['Δnet (mm/mo)'].mean():.3f}",
        f"{mid['Δnet (mm/mo)'].min():.3f}",
        f"{(mid['Δnet (mm/mo)'].abs() > mid['Hist Avg net (mm/mo)'].abs()).sum()} of {len(mid)}",
    ],
})
state_agg.to_csv(f'{FIG_DIR}/_statewide_aggregate.csv', index=False)
print('\nStatewide aggregate (mid WUE):')
display(state_agg)

,HUC12,Watershed,n_facilities,Total MW,WUE,Cons MGD,Cons mm/mo,Hist Avg net (mm/mo),Δnet (mm/mo),Δnet CI lo,Δnet CI hi
14,160300050805,Sugarville,3,10050.0,high,102.0,156.012,-15.58,-46.588,-55.554,-37.622
12,160300050805,Sugarville,3,10050.0,low,25.5,39.003,-15.58,-11.647,-13.888,-9.406
13,160300050805,Sugarville,3,10050.0,mid,76.5,117.009,-15.58,-34.941,-41.665,-28.217
11,160203090501,Headwaters Hansel Valley Wash,2,9000.0,high,91.3,71.782,0.47,-21.435,-25.561,-17.310
9,160203090501,Headwaters Hansel Valley Wash,2,9000.0,low,22.8,17.945,0.47,-5.359,-6.390,-4.328
10,160203090501,Headwaters Hansel Valley Wash,2,9000.0,mid,68.5,53.836,0.47,-16.077,-19.170,-12.983
17,160300051502,Whiskey Creek,1,4000.0,high,40.6,39.269,-1.29,-11.726,-13.983,-9.470
15,160300051502,Whiskey Creek,1,4000.0,low,10.1,9.817,-1.29,-2.932,-3.496,-2.367
16,160300051502,Whiskey Creek,1,4000.0,mid,30.4,29.451,-1.29,-8.795,-10.487,-7.102
8,160202040106,Bingham Creek,2,147.0,high,1.5,2.147,14.04,-0.641,-0.764,-0.518



Statewide aggregate (mid WUE):


,Metric,Value
0,Total planned MW,"23,222"
1,Total cooling water (MGD),176.7
2,"Avg watershed Δnet (mm/mo, mid WUE)",-10.063
3,"Worst watershed Δnet (mm/mo, mid WUE)",-34.941
4,Watersheds with |Δnet| > |Hist Avg|,3 of 6


### Figure 11: Cumulative Per-Watershed Impact Bar Chart

A horizontal bar chart of projected Δ net availability per watershed at mid WUE, with cluster-robust 95% CI error bars. Three watersheds dominate: Sugarville (−34.9 mm/mo), Headwaters Hansel Valley Wash (−16.1 mm/mo, Stratos), and Whiskey Creek (−8.8 mm/mo). The remaining three watersheds (Bingham Creek, Summit Creek, Big Dry Creek) host small operating expansions with minimal projected impact.

In [114]:
# Figure 11 — cumulative per-watershed bar chart at mid WUE
# Sorted ascending (most-negative at bottom) so the worst-case watersheds are visually dominant
mid_plot = mid.copy().sort_values('Δnet (mm/mo)')
y = np.arange(len(mid_plot))

fig11, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(y, mid_plot['Δnet (mm/mo)'],
        color='#c0392b', alpha=0.85,
        # Error bars represent the cluster-robust 95% CI from the OLS coefficient
        xerr=[mid_plot['Δnet (mm/mo)'] - mid_plot['Δnet CI lo'],
              mid_plot['Δnet CI hi']    - mid_plot['Δnet (mm/mo)']],
        ecolor='black', capsize=4, edgecolor='black', linewidth=0.5)

# Label each bar with watershed name + total MW for stakeholder context
ax.set_yticks(y)
ax.set_yticklabels(
    [f"{r['Watershed'][:26]} ({int(r['Total MW']):,} MW)" for _, r in mid_plot.iterrows()],
    fontsize=9
)
ax.axvline(0, color='black', linewidth=0.6)
ax.set_xlabel('Projected Δ Net Availability (mm/month) — cluster-robust 95% CI')
ax.set_title('Cumulative Per-Watershed Impact of Proposed Utah Data Center Pipeline (mid WUE = 1.5 L/kWh)')
plt.savefig(f'{FIG_DIR}/11_cumulative_impact.png')
plt.close()

### Final Touches Summary

This section added:

- **11 figures** (saved to `figures/`) covering distribution, time series, scatter, correlation, two spatial maps, predicted-vs-actual, residual diagnostics, coefficient forest, and cumulative impact bar.
- **A leave-driest-year-out robustness check** confirming the thermoelectric coefficient is stable to extreme-drought removal (change ≈ +0.4%).
- **A cumulative pipeline analysis** extending Milestone 3's three-watershed projection to all six watersheds hosting planned facilities, with a statewide aggregate (23,222 MW pipeline → 177 MGD cooling demand → 6 watersheds affected, worst-case Sugarville at −35 mm/month net availability impact).
- **A limitations / additional data section** stating what would improve precision without altering qualitative conclusions.

The model is ready for deployment as a **scenario planning tool** for Utah water resource managers and proposed-facility siting reviews. It should not be used as a sole source of facility approval — the 36 thermoelectric training observations limit precision and the model cannot resolve sub-watershed dynamics — but it provides a defensible, statistically rigorous baseline for understanding the order of magnitude of impact and the relative ranking of risk across proposed sites.

# Writeup

## The Problem
Water scarcity is one of the defining resource challenges of the American West. Utah, an arid state that depends heavily on snowpack-fed rivers and reservoirs, is particularly vulnerable. At the same time, Utah has emerged as a rapidly growing technology and energy hub: tax incentives, available land, and access to power have attracted a pipeline of over 23 gigawatts of proposed data center capacity, with a single project (Stratos in Box Elder County) targeting 9 GW. Nuclear small modular reactors are also being revisited as a low-carbon baseload option, with proposed sites in the region. Both data centers and nuclear power plants are significant industrial water consumers: nuclear plants use water for steam generation and cooling, while data centers rely on evaporative cooling that can consume millions of gallons per day at scale.

This project asks the following question: **To what extent does thermoelectric water consumption from nuclear power plants and large industrial facilities such as data centers reduce effective water availability in Utah watersheds, and can we predict the impact of future growth in that consumption?**

## Why It Matters
Utah’s pipeline of proposed data centers is roughly 25 times the state’s current operational data center capacity. The largest proposed sites — Sugarville (10,050 MW across three facilities), Headwaters Hansel Valley Wash (9,000 MW Stratos), and Whiskey Creek (4,000 MW HPC Campus) — all sit in endorheic Great Basin watersheds with no downstream outlet. Every unit of consumptive water loss in these basins is permanent. Yet only 8 of the 17 catalogued facilities have publicly reported water consumption figures, leaving water managers, state regulators, and downstream communities without a defensible basis for sustainable siting decisions. A quantitative model that links thermoelectric demand to watershed-level water availability fills exactly this gap.

## EDA
The watershed panel covers 25,580 observations (2,558 watersheds × 10 years). The distribution of net availability is sharply skewed: most watershed-years are slightly water-positive, but a long left tail extends well below zero, reflecting the structural water deficit in arid central and western Utah. As shown in Figure 1, 8.9% of all observations are in deficit even before the proposed data center pipeline is operational.

<figure>
  <img src="figures/01_net_avail_distribution.png" alt="Figure 1" style="max-width:100%"/>
  <figcaption>Figure 1. Distribution of net water availability across all 25,580 Utah HUC12 watershed-year observations, 2010–2019. The dashed line marks the stress threshold (net = 0). 8.9% of observations are already in deficit before any proposed buildout.</figcaption>
</figure>

The statewide hydrology time series (Figure 2) shows that consumption is essentially flat across the decade (red squares hover near 0.5–0.6 mm/mo) while natural availability varies by a factor of five between wet years (2011 peak ≈ 8.5 mm/mo) and dry years (2018 trough ≈ 1.7 mm/mo). Net availability therefore tracks natural availability almost perfectly at the statewide aggregate scale. The thermoelectric effect — small in absolute terms — only becomes detectable when the watershed-level fixed-effects estimator isolates within-watershed year-to-year variation.

<figure>
  <img src="figures/02_statewide_hydrology.png" alt="Figure 2" style="max-width:100%"/>
  <figcaption>Figure 2. Statewide annual mean natural availability, total consumption, and net availability across 2,558 HUC12 watersheds, 2010–2019. 2018 is the driest year in the dataset; it falls in the held-out test set.</figcaption>
</figure>

A scatter of natural availability against net availability (Figure 3) makes the consumptive depression visible at the observation level. The non-thermoelectric majority of watershed-years cluster tightly along the y = x reference, indicating that net availability simply equals natural availability where human consumption is small. The thermoelectric-active watershed-years — roughly fifty observations corresponding to the three operating coal-plant watersheds in Emery County — deviate downward from this baseline. This is the empirical signal the OLS model extracts as a coefficient.

<figure>
  <img src="figures/05_avail_vs_net_scatter.png" alt="Figure 3" style="max-width:100%"/>
  <figcaption>Figure 3. Natural availability versus net availability for all watershed-year observations. Non-thermoelectric (gray) watersheds align with the y = x reference; thermoelectric-active (red) watersheds deviate downward, displaying the consumptive depression that the regression model quantifies.</figcaption>
</figure>

## Data Preparation

### Unified Watershed Panel
Utah’s electricity generation mix has changed substantially over the 2000–2025 horizon (Figure 4). Coal generation has fallen from over 80% of statewide net generation to under 60%; natural gas has roughly tripled; and renewable generation (solar, wind, geothermal) has grown from negligible to a measurable share. The model’s `fossil_fraction` feature captures this composition shift, and the cluster-robust OLS coefficient on `fossil_fraction` is −0.44 mm/mo (95% CI [−0.59, −0.28], p < 0.001) — statewide shifts toward fossil generation depress net availability.

<figure>
  <img src="figures/04_generation_mix.png" alt="Figure 4" style="max-width:100%"/>
  <figcaption>Figure 4. Utah electricity generation by fuel type, 2000–2025 (UofU Energy Landscape Report). Coal has declined; natural gas and renewables have grown.</figcaption>
</figure>

The base panel was constructed by merging the USGS National Water Availability Assessment outputs (one row per HUC12 × year, fields including `availab_mm/mo` and `consum_mm/mo`) with the USGS thermoelectric water-use model (fields `tewdftot_mgd` for withdrawal and `tecuftot_mgd` for consumption). Thermoelectric records were available only for watersheds containing active facilities; missing rows were filled with zero. Watershed area in km² was joined from the HUC12 boundary GeoPackage downloaded via the `pygeohydro` WBD client.
The UofU Energy Landscape dataset provided statewide-annual features that vary in time but are constant across watersheds within a year: `renewable_fraction`, `fossil_fraction`, `industrial_gwh`, and `industrial_yoy_growth`. These were merged onto the watershed panel by year. The PacifiCorp IRP Table 7.4 supplied design-specification water intensities (Gal/MWh) used to (a) validate the UofU × intensity estimate of statewide thermoelectric consumption against USGS actuals (Figure 5) and (b) construct projection scenarios for future technology mixes (nuclear SMR at 720 Gal/MWh, natural gas CCCT at 210, etc.).

<figure>
  <img src="figures/03_thermo_validation.png" alt="Figure 5" style="max-width:100%"/>
  <figcaption>Figure 5. Validation of the UofU × intensity-factor estimate of statewide thermoelectric water consumption against the USGS NWAA actuals. Relative shape agreement validates the intensity-factor methodology used in projection scenarios.</figcaption>
</figure>

### Unit Normalization
The target variable is `net_avail_mm_mo = availab_mm/mo − consum_mm/mo`, expressed in mm/month depth. The original USGS thermoelectric variables `tecuftot_mgd` and `tewdftot_mgd` are flow rates in million gallons per day — incommensurable with the target depth unit without watershed area. This unit mismatch is one of the most consequential structural decisions in the analysis. The transformation is:

$ thermo_consum_mm_mo = tecuftot_mgd × (365.25 / 12) × 3.785 × 10⁻³ × 1000 / areasqkm ≈ tecuftot_mgd × 115.22 / areasqkm $

The 10⁶ gal/MG and 10⁶ m²/km² factors cancel; the result is a depth-per-month rate directly comparable to the target. For Utah’s active thermoelectric watersheds (median area 85.8 km²), the resulting depths range from 0.54 to 3.20 mm/mo — small relative to natural availability fluctuations, which motivates the within-watershed fixed-effects estimator used in Milestone 3.
A correlation heatmap of the demeaned features (Figure 6) confirms no severe multicollinearity remains after the fixed-effects transformation. The high correlation between `availab_mm_mo` and `net_avail_mm_mo` (r ≈ 1.0) is expected and benign: it reflects the dominant physical relationship that net availability tracks natural availability when consumption is held constant. The model’s job is to extract the small thermoelectric departure from that baseline.

<figure>
  <img src="figures/06_feature_correlation.png" alt="Figure 6" style="max-width:100%"/>
  <figcaption>Figure 6. Post-fixed-effects feature correlation matrix (training set). No predictor pair exhibits |r| > 0.9 except for the target’s expected positive correlation with natural availability.</figcaption>
</figure>

### Temporal Train/Test Split and Scaling
Because the panel includes a temporal dimension, random k-fold splitting would leak future information into training (a model could see 2019 in training and 2017 in test, predicting the past from the future). The split was therefore made on time: 2010–2017 to training (20,464 rows) and 2018–2019 to test (5,116 rows). The 2018 test year is the driest in the dataset and 2019 is a wet rebound, so the test set spans contrasting hydrological conditions.
Continuous features were standardized (zero mean, unit variance) using a StandardScaler fitted on the training set only. The same fitted scaler was then applied (transform-only) to the test set. This anti-snooping protocol prevents test-set distribution information from leaking into the training pipeline.

### Watershed Fixed-Effects Demeaning
The panel of 2,558 watersheds × 10 years contains substantial unobserved time-invariant heterogeneity. Geology, elevation, slope, land cover, and existing infrastructure differ across watersheds in ways the available features cannot fully describe; some of these time-invariant attributes are also correlated with thermoelectric siting decisions (coal plants were sited near coal seams and rivers; data centers cluster near power infrastructure). Within-watershed demeaning — subtracting each watershed’s training-set mean from every observation of that watershed — removes time-invariant confounders by construction. The remaining identification is from year-to-year variation within each watershed.
After demeaning, the coefficient on `thermo_consum_mm_mo` measures the within-watershed effect of additional thermoelectric consumption: precisely the estimand of interest for projecting impacts of new facilities. Test observations were demeaned using the training-set watershed means to prevent any test-set leakage.

## Model Building and Evaluation

### Models Trained
Three models were trained on the watershed fixed-effects demeaned panel:
- Linear Regression (OLS with watershed fixed effects). Interpretable; the `thermo_consum_mm_mo` coefficient is a direct within-watershed estimate.
- Ridge Regression with leave-one-out CV alpha selection (α = 0.1). Acts as an L2-regularized stability check on the sparse thermoelectric signal.
- Random Forest (300 trees, max_depth 10, min_samples_leaf 100). Captures any non-linear threshold effects (e.g., stress amplification when natural availability is near zero).

### Cross-Validation Strategy
GroupKFold with 5 folds and `huc12_id` as the group key was used for model selection. Each validation fold contains watersheds entirely absent from the training fold. This directly tests the deployment scenario: predicting net availability impacts in proposed-facility watersheds that have no thermoelectric history. Standard random k-fold would yield optimistic CV metrics because the model would see watersheds during training. GroupKFold CV RMSE was 0.221 ± 0.035 mm/mo for both OLS and Ridge.

### Test-Set Evaluation
OLS and Ridge produced essentially identical test-set performance (RMSE = 0.326 mm/mo, R² = 0.994). Figure 7 shows the predicted-vs-actual scatter on both partitions: tight diagonal alignment confirms the model fit quality. The slight scatter visible at the bottom-left of the test panel corresponds to 2018 drought observations, where the model conservatively underestimates how dry the year was — a recognized limitation discussed below.

<figure>
  <img src="figures/08_pred_vs_actual.png" alt="Figure 7" style="max-width:100%"/>
  <figcaption>Figure 7. Predicted vs. actual net availability (demeaned) for the OLS fixed-effects model on the training set (left) and held-out test set (right).</figcaption>
</figure>

Random Forest performed substantially worse (test RMSE = 2.46, R² = 0.67). With only 36 thermoelectric watershed-years in training (out of 20,456), the forest cannot construct splits on `thermo_consum_mm_mo` at the `min_samples_leaf = 100` threshold required to avoid overfitting. The linear models are clearly preferred for this problem.

### Statistical Inference
OLS standard errors assume independent residuals, but observations within the same watershed share hydrology, geology, and infrastructure. Cluster-robust standard errors clustered by `huc12_id` correct for this. The cluster-robust 95% confidence interval for the key coefficient is [−0.356, −0.241]. An independent pairs-bootstrap with 500 resamples gave [−0.358, −0.260], confirming the cluster-robust interval.

<figure>
  <img src="figures/10_coefficient_forest.png" alt="Figure 8" style="max-width:100%"/>
  <figcaption>Figure 8. Cluster-robust 95% confidence intervals for the five OLS fixed-effects coefficients. `thermo_consum_mm_mo` is unambiguously negative, with the upper CI bound at −0.241, far from zero.</figcaption>
</figure>

Quantile regression at τ = 0.10 (the dry-year tail) yields a `thermo_consum_mm_mo` coefficient of −0.328 versus −0.299 at the conditional mean — a 10% amplification of thermoelectric stress in dry years. The `fossil_fraction` coefficient amplifies 3.6× from −0.44 at the mean to −1.58 at τ = 0.10. Both findings confirm that thermoelectric and energy-mix impacts are disproportionately concentrated in the years when water is already scarcest.

### Robustness Check
A leave-driest-training-year-out check refits the model after removing 2012 (the lowest-availability training year, mean 2.68 mm/mo). The `thermo_consum_mm_mo` coefficient changes from −0.299 to −0.297 — a +0.4% change well within the 95% CI of the full estimate. The thermoelectric coefficient does not depend on a single drought year.

### Residual Diagnostics

<figure>
  <img src="figures/09_residual_diagnostics.png" alt="Figure 9" style="max-width:100%"/>
  <figcaption>Figure 9. Test-set residual diagnostics. Left: residuals vs. fitted. Right: Q-Q plot against a normal distribution; the heavy left tail reflects the model’s conservative under-estimation of how dry 2018 was.</figcaption>
</figure>

Test residuals have mean 0.006 and standard deviation 0.326, indicating overall unbiased prediction. The Q-Q plot shows a heavy left tail (skew = −6.80), consistent with the model under-estimating the severity of the 2018 drought. This is a known limitation: no training year matches 2018’s aridity, so the model treats it as an extreme extrapolation. For projection purposes, this means the model is conservative on drought risk — the actual impact of data center demand in a drought year is likely to be modestly larger than the projection implies.

### Projection Methodology
For projection scenarios, the unscaled OLS coefficient is applied directly. Each proposed facility’s capacity (MW) is converted to cooling water demand (MGD) using the project’s standard helper `dc_water_mgd(mw, wue)` with a default mid-range Water Usage Effectiveness of 1.5 L/kWh (low 0.5 and high 2.0 also computed). The MGD is then converted to mm/month depth using the watershed area, multiplied by the OLS coefficient (−0.299), and the cluster-robust 95% CI bounds [−0.356, −0.241] are propagated through the same multiplication to give an uncertainty interval on the projected impact.

For watersheds hosting multiple proposed facilities, demand is summed before conversion. This produces the cumulative per-watershed projection presented in the next section.

## Results: Projecting Future Demand

### Current vs. Projected Spatial Distribution
The most striking visualization in this project is the side-by-side map comparison of current and projected watershed consumption (Figure 10). Today, consumption is broadly distributed across Utah’s populated and agricultural areas, with modest concentrations along the Wasatch Front and in Emery County (existing coal-plant watersheds). After the proposed pipeline is operational, three Great Basin watersheds — Sugarville, Headwaters Hansel Valley Wash, and Whiskey Creek — emerge as new extreme hotspots, each absorbing an order of magnitude more cooling water than any current site.

<figure>
  <img src="figures/07_maps_current_vs_projected.png" alt="Figure 10" style="max-width:100%"/>
  <figcaption>Figure 10. Utah HUC12 watersheds colored by water consumption rate (mm/month). Map A shows the 2010–2019 historical mean; triangles mark existing thermoelectric watersheds. Map B shows the projected post-buildout consumption at mid-WUE (1.5 L/kWh); circles size-by-MW show proposed data center locations. Both maps share a logarithmic SymLog color scale.</figcaption>
</figure>

### Cumulative Per-Watershed Impact
Six watersheds host planned facilities in the Cleanview inventory. Aggregating facility-level cooling demand within each watershed and propagating through the OLS coefficient yields per-watershed projected reductions in net availability (Figure 11). The three large Great Basin sites dominate; the other three watersheds (Bingham Creek, Summit Creek, Big Dry Creek) host smaller expansions with minimal projected impact.
<figure>
  <img src="figures/11_cumulative_impact.png" alt="Figure 11" style="max-width:100%"/>
  <figcaption>Figure 11. Cumulative projected change in net availability per watershed at mid WUE (1.5 L/kWh), with cluster-robust 95% confidence intervals from the OLS coefficient. Labels show watershed name and total planned MW capacity.</figcaption>
</figure>

### Statewide Aggregate
The total planned pipeline is 23,222 MW across the six watersheds, corresponding to a cumulative cooling-water demand of approximately 177 MGD at mid-WUE. This is roughly four times the entire current statewide thermoelectric consumption baseline of ~43 MGD. The mid-WUE worst-case watershed is Sugarville, projected to lose 34.9 mm/month of net availability (cluster-robust 95% CI [−41.7, −28.2]); the historical average for that watershed is already −15.6 mm/month, so the combined post-buildout average would be approximately −50 mm/month. Headwaters Hansel Valley Wash (Stratos) is currently barely water-positive at +0.5 mm/month average, but the proposed 9,000 MW facility would drive it to approximately −16 mm/month — a sign change in the basin’s water balance.

## Conclusion

### What the Model Tells Us
The cluster-robust within-watershed coefficient on thermoelectric consumption is −0.299 mm/month per mm/month, with a 95% confidence interval that excludes zero by a comfortable margin ([−0.356, −0.241]). The pairs-bootstrap confirmation ([−0.358, −0.260]) provides an independent check via a different statistical mechanism. The robustness check (leave-driest-year-out) confirms the coefficient is not driven by extreme-drought training observations. Quantile regression at the drought tail shows the effect amplifies modestly (~10%) in dry years and the energy-mix effect amplifies dramatically (~3.6×). The model generalizes to never-thermoelectric watersheds (OOS test RMSE = 0.32 mm/month, R² = 0.994 across 5,098 observations) — the exact deployment scenario for projection.

When the coefficient is applied to the 23,222 MW Utah data center pipeline at mid-WUE, three Great Basin watersheds emerge as severe stress hotspots: Sugarville (−34.9 mm/mo), Headwaters Hansel Valley Wash / Stratos (−16.1 mm/mo), and Whiskey Creek (−8.8 mm/mo). All three are endorheic with no downstream outlet; consumptive loss in these basins is permanent. The remaining three planned-pipeline watersheds have minor impact, but the headline finding is unambiguous: the proposed pipeline produces large, statistically significant water stress concentrated in a small number of closed-basin watersheds that are not currently in deficit.

### Is the Model Ready to Be Deployed?
The model is ready for use as a scenario planning tool for water resource managers and as input to facility-siting reviews. Its strengths are clear: it is built on the authoritative USGS NWAA dataset, uses correct unit normalization and a watershed-fixed-effects estimator that identifies a causally interpretable within-watershed effect, reports uncertainty via two independent statistical methods, has been stress-tested against a leave-one-year-out robustness check, generalizes to unseen watersheds, and produces a directly usable mm/mo-per-mm/mo coefficient that can be applied to any new facility for which a cooling-water demand can be estimated.

The model should not, however, be the sole basis for facility approval or denial. The training set contains only 36 thermoelectric watershed-years, all at existing Utah coal plants. Extrapolating to data-center evaporative cooling assumes the same physical mechanism applies, which is reasonable but not directly validated. Sub-watershed dynamics (groundwater recharge, return flows, intra-month variability) are not resolved by the HUC12 × year panel. The model is therefore best understood as a defensible baseline for understanding the order of magnitude of impact and the relative ranking of risk across proposed sites — not as a substitute for site-specific hydrological studies.

### Recommendations
- For Utah water resource managers: prioritize watershed-level water-availability studies for Sugarville (HUC12 160300050805), Headwaters Hansel Valley Wash (160203090501), and Whiskey Creek (160300051502) before approving the proposed facilities in those basins. The combined impact of multiple facilities in Sugarville is the single largest projected risk in the pipeline.
- For state regulators: require facility-level water consumption measurement and public reporting for all proposed and operating data centers. The current 9-of-17 reporting gap leaves regulators without a basis for calibrating projection models or enforcing water-use conditions.
- For technology companies conducting water impact assessments: use the model’s mm/mo-per-mm/mo coefficient with cluster-robust 95% CI as a defensible projection framework. Site-specific WUE values should replace the default 1.5 L/kWh assumption where measured cooling-system performance data is available.
- For environmental planners: recognize that the endorheic nature of the high-impact Great Basin watersheds means the consumptive losses are permanent and irreversible. Mitigation through return-flow recovery is not available in closed basins.

### Limitations and Additional Data That Would Improve Accuracy

The model produces statistically robust within-watershed estimates with a tightly bounded 95% CI, and projections track expected physical magnitudes. Several data gaps still constrain confidence in the projection numbers:

1. **Facility-level measured water consumption.** WRA (2025) reports actual water use for only 8 of 17 catalogued operating data centers. Verified consumption data for the remaining 9, plus mandatory reporting for proposed facilities, would replace the WUE-based intensity assumption (currently 0.5–2.0 L/kWh range, midpoint 1.5) with directly measured values.
2. **Post-2019 USGS NWAA observations.** The watershed model panel ends in 2019. Data center industrial load roughly doubled in Utah between 2019 and 2025; the most recent thermoelectric responses to that growth are not in the training set. Re-extracting NWAA 2020–2025 outputs when the USGS releases them would strengthen both the historical coefficient estimate and the calibration of future projections.
3. **Groundwater and aquifer recharge data.** The target `net_avail_mm_mo` captures surface water; the Great Basin watersheds (Sugarville, Hansel Valley, Whiskey Creek) where the largest proposed facilities will operate are endorheic and partly groundwater-dependent. USGS National Water Information System (NWIS) well data per HUC12 would allow extending the analysis to total water balance.
4. **Climate projections for natural availability.** Future scenarios in this notebook hold `availab_mm_mo` at its historical mean. Coupling to downscaled IPCC CMIP6 projections for the Utah/Great Basin region would quantify how climate-driven declines in availability *compound* with data-center demand growth.
5. **Return-flow accounting.** OLS implicitly assumes all `thermo_consum_mm_mo` is fully consumptive. Some cooling-tower configurations have measurable blowdown that returns to nearby surface water. Facility-level return-flow audits would refine the within-watershed coefficient by separating consumption from net withdrawal.

These gaps do **not** alter the qualitative conclusion (proposed pipeline produces large, statistically unambiguous reductions in net availability in three Great Basin watersheds). They affect the precision of the absolute numbers, not the direction or significance.

## Data Sources

- United States Geological Survey (USGS):
  - [Model: Thermoelectric Power Water-Use Model](https://water.usgs.gov/nwaa-data/data-catalog/sector/water-use/model/wu-thermoelectric/)
  - [Model: National Water Availability Assessment Outputs (CONUS 2025)](https://water.usgs.gov/nwaa-data/data-catalog/sector/integrated-water-availability/model/iwa-assessment-outputs-conus-2025/)
- Western Resource Advocates (WRA):
  - [Data Center Impacts in the West: Policy Solutions for Water and Energy Use](https://westernresourceadvocates.org/wp-content/uploads/2025/08/2025_DataCenter_FactSheet_UT.pdf)
- Salt Lake Tribune (SLT):
  - [Can Utah become a data center hub without draining its water supply?](https://www.sltrib.com/news/politics/2026/01/12/does-utah-have-enough-water-be/)
- Stanford University:
  - [Nuclear Reactor Water Usage and the Implications of Limited Water Availability](http://large.stanford.edu/courses/2013/ph241/abdul-khabir2/)
- University of Utah (UofU):
  - [Utah’s 2025 Energy Landscape and Outlook for 2026 and Beyond](https://d36oiwf74r1rap.cloudfront.net/wp-content/uploads/2026/04/ERG2026-Energy-RB-Apr2026.pdf)
- PacifiCorp:
  - [2025 Integrated Resource Plan: Volume 1](https://www.pacificorp.com/content/dam/pcorp/documents/en/pacificorp/energy/integrated-resource-plan/2025-irp/2025_IRP_Vol_1.pdf)
- Utah Clean Energy Organization (UCE):
  - [Estimated Emissions and Water Consumption from the Proposed Stratos Data Center](https://utahcleanenergy.org/estimated-emissions-and-water-consumption-from-the-proposed-stratos-data-center/)
- Deseret News
  - [Data centers on the way to Utah, but how fast and at what cost? ](https://www.deseret.com/business/2026/04/15/utah-data-centers-artificial-intelligence-ai-water-energy-electricity-use-economic-impacts-jobs-tax-incentives/)
- ABC4 News
  - [Tracking where AI data centers are being built in Utah: Interactive map](https://www.abc4.com/news/digital-exclusives/tracking-ai-data-centers-utah-map/)